# Nhận diện mũ bảo hộ lao động trên Google Colab

Notebook này chỉ chạy trên runtime Colab có GPU. Không cài hoặc chạy huấn luyện trên máy cá nhân.

## Quy tắc dữ liệu và lớp mô hình

JSON rà soát của browser giữ ontology review tám lớp để phân biệt mũ bảo hộ, mũ giao thông, mũ lưỡi trai, mũ len, đầu trần, mũ trong hình, mũ rời và vùng poster. Converter chỉ đưa hai lớp vào YOLO: `hard_hat_worn` (model ID 0) là mục tiêu; `depiction_region` (model ID 1) là lớp phụ để loại người nằm trong poster/ảnh/màn hình. Các nhãn review khác không được xuất thành lớp mô hình. Ảnh chỉ có mũ giao thông, mũ lưỡi trai, mũ len hoặc cảnh nền được giữ với tệp `.txt` rỗng làm ví dụ âm tính.

Poster/ảnh in/màn hình vẫn phải được khoanh đủ. Hardhat nằm trong vùng đó không được gán thành hardhat thật; overlap được suppression/uncertain theo chính sách hiện tại và không bao giờ tự tăng `wearer_count`. Nếu vùng hiển thị bị bỏ sót, borderless hoặc chưa xác minh, không dùng ảnh đó như ảnh nền âm tính đã rà.

Các lớp mũ âm `traffic_helmet_worn`, `cap_worn` và `beanie_worn` đều cần tối thiểu 300 group độc lập ở train, 30 ở validation và 100 ở locked test. Gom ảnh có cùng người được nhận diện, cùng video/phiên chụp, cùng nội dung hiển thị hoặc cùng ảnh nguồn (kể cả crop và ảnh gần trùng) vào cùng một `group_id`; một group chỉ thuộc một split. Chỉ tính group có subtype mũ đội được người rà xác nhận và ghi trong `hat_type_negative_sources` khớp nhãn rà. Nếu chưa chắc tính độc lập, không tính group vào quota. Notebook kiểm tra riêng từng subtype ở từng split và chặn preflight/release nếu thiếu bất kỳ sàn nào.

## Chuẩn bị dữ liệu trên Drive

Tạo `MyDrive/helmet_project/datasets/hardhat_only_v1.zip`. ZIP chứa `data.yaml`, ảnh và nhãn cho ba split `train`, `val`, `test`. Mọi ảnh phải có tệp nhãn; ảnh nền âm tính đã rà dùng tệp `.txt` rỗng.

Trước khi staging dữ liệu trong Colab, Drive phải có thêm `test_scene_metadata.csv` và `hard_negative_source_groups.csv` đã rà/confirm. Cell chuẩn bị kiểm tra đủ cả ba tệp trước khi copy hoặc giải nén ZIP; metadata `dataset_image_sources.json` phải khai báo `training_use=approved` và đủ trường quyền theo từng ảnh. Đây là kiểm tra schema/declaration, không tự xác minh quyền với chủ nguồn.

Dùng ảnh và clip có quyền sử dụng. Chia tập theo người, video, cảnh và mẫu poster. Giữ test tách biệt hoàn toàn khỏi train và validation. Gán thử, rà lỗi và khóa quy tắc trước khi gán cả tập.


In [ ]:
import os
if not os.environ.get("COLAB_RELEASE_TAG"):
    raise RuntimeError(
        "Google Colab managed runtime required. Do not run this training notebook on local Jupyter."
    )
from google.colab import drive
drive.mount("/content/drive")

%pip -q install "ultralytics==8.4.202" pyyaml


In [ ]:
from pathlib import Path, PurePosixPath
import csv
import hashlib
import json
import shutil
import zipfile
import yaml

DRIVE_ROOT = Path("/content/drive/MyDrive/helmet_project")
ARCHIVE_DRIVE = DRIVE_ROOT / "datasets" / "hardhat_only_v1.zip"
SCENE_METADATA_CSV = DRIVE_ROOT / "datasets" / "test_scene_metadata.csv"
HARD_NEGATIVE_SOURCE_GROUPS_CSV = DRIVE_ROOT / "datasets" / "hard_negative_source_groups.csv"
HARD_NEGATIVE_SOURCE_TYPES = ("poster", "printed_photo", "screen", "loose_hat", "mixed_scene", "empty_background")
HAT_TYPE_NEGATIVE_SOURCES = ("traffic_helmet_worn", "cap_worn", "beanie_worn")
REVIEW_CLASS_NAMES = ("hard_hat_worn", "traffic_helmet_worn", "cap_worn", "beanie_worn", "bare_head", "depicted_hat", "loose_hat", "depiction_region")
REVIEWED_SURFACE_TYPES = {"poster", "printed_photo", "screen", "other_depiction"}
WORK_ROOT = Path("/content/hardhat_only_v1")
ARCHIVE_LOCAL = WORK_ROOT / "hardhat_only_v1.zip"
EXTRACT_ROOT = WORK_ROOT / "extracted"
DRIVE_RUNS = DRIVE_ROOT / "runs" / "hardhat_only"
RUN_NAME = "hardhat_only_v1"
DRIVE_RUN = DRIVE_RUNS / RUN_NAME
LOCAL_RUNS = WORK_ROOT / "runs"

REQUIRED_DRIVE_INPUTS = {
    "reviewed dataset ZIP": ARCHIVE_DRIVE,
    "reviewed test scene metadata CSV": SCENE_METADATA_CSV,
    "confirmed hard-negative source-group CSV": HARD_NEGATIVE_SOURCE_GROUPS_CSV,
}
missing_drive_inputs = [
    f"{label}: {path}"
    for label, path in REQUIRED_DRIVE_INPUTS.items()
    if not path.is_file()
]
if missing_drive_inputs:
    raise FileNotFoundError(
        "Colab preflight refused to stage data; missing required reviewed inputs: "
        + "; ".join(missing_drive_inputs)
    )

WORK_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ARCHIVE_DRIVE, ARCHIVE_LOCAL)
with ARCHIVE_LOCAL.open("rb") as archive_stream:
    archive_digest = hashlib.sha256()
    for chunk in iter(lambda: archive_stream.read(1 << 20), b""):
        archive_digest.update(chunk)
archive_sha256 = archive_digest.hexdigest()
print("Dataset ZIP SHA-256:", archive_sha256)

# Reject archive paths that would write outside the extraction root.
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE_LOCAL) as archive:
    root = EXTRACT_ROOT.resolve()
    for member in archive.infolist():
        target = (root / PurePosixPath(member.filename)).resolve()
        if not target.is_relative_to(root):
            raise ValueError(f"Unsafe ZIP member path: {member.filename}")
    archive.extractall(EXTRACT_ROOT)

yaml_files = list(EXTRACT_ROOT.rglob("data.yaml"))
if len(yaml_files) != 1:
    raise ValueError(f"Expected exactly one data.yaml inside the archive, found {len(yaml_files)}")
DATASET_ROOT = yaml_files[0].parent.resolve()


def reject_duplicate_json_keys(pairs):
    value = {}
    for key, item in pairs:
        if key in value:
            raise ValueError(f"Duplicate JSON key: {key}")
        value[key] = item
    return value


def parse_source_list(row, field, allowed, where):
    try:
        values = json.loads(row.get(field, ""))
    except (TypeError, json.JSONDecodeError) as error:
        raise ValueError(f"{where}: {field} must be a JSON array; [] is an explicit reviewed empty value") from error
    if (not isinstance(values, list)
            or any(not isinstance(value, str) or value not in allowed for value in values)
            or len(values) != len(set(values))):
        raise ValueError(f"{where}: {field} must be a unique JSON array from {sorted(allowed)}")
    return values


def load_hard_negative_source_groups(dataset_root, groups_csv):
    manifest_path = dataset_root / "split_manifest.csv"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Missing split manifest: {manifest_path}")
    assignments = {}
    group_profiles = {}
    folded_groups = {}
    with manifest_path.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if tuple(reader.fieldnames or ()) != ("image_id", "split", "group_id", "scene_type"):
            raise ValueError("split_manifest.csv must contain image_id,split,group_id,scene_type")
        for line_number, row in enumerate(reader, 2):
            image_id = (row.get("image_id") or "").strip()
            split = (row.get("split") or "").strip()
            group_id = (row.get("group_id") or "").strip()
            scene_type = (row.get("scene_type") or "").strip()
            if (not image_id or not group_id or split not in {"train", "val", "test"}
                    or scene_type not in {"depiction_only", "mixed_scene", "other"}
                    or image_id != row.get("image_id") or group_id != row.get("group_id")
                    or any(ord(char) < 32 for char in image_id + group_id)):
                raise ValueError(f"split_manifest.csv:{line_number} has invalid image/split/group/scene metadata")
            if image_id in assignments:
                raise ValueError(f"split_manifest.csv duplicates image_id {image_id!r}")
            folded = group_id.casefold()
            previous = folded_groups.setdefault(folded, (group_id, split))
            if previous != (group_id, split):
                raise ValueError(f"group_id {group_id!r} is duplicated with changed case or crosses splits")
            assignments[image_id] = (split, group_id, scene_type)
            key = (split, group_id)
            profile = group_profiles.setdefault(key, {
                "scene_types": set(),
                "surface_types": set(),
                "image_review_statuses": set(),
                "review_class_counts": {name: 0 for name in REVIEW_CLASS_NAMES},
            })
            profile["scene_types"].add(scene_type)

    if not assignments:
        raise ValueError("split_manifest.csv contains no image groups")
    source_path = dataset_root / "dataset_image_sources.json"
    if not source_path.is_file():
        raise FileNotFoundError(f"Missing image-source metadata: {source_path}")
    image_sources = json.loads(source_path.read_text(encoding="utf-8"), object_pairs_hook=reject_duplicate_json_keys)
    if (not isinstance(image_sources, dict)
            or image_sources.get("format") != "helmet-vision-image-sources-v4"
            or image_sources.get("review_format") != "helmet-vision-review-v7"
            or not isinstance(image_sources.get("images"), list)):
        raise ValueError("dataset_image_sources.json must use image-sources-v4 for review-v7 with an images array")
    seen_source_images = set()
    for index, record in enumerate(image_sources["images"]):
        where = f"dataset_image_sources.json images[{index}]"
        if not isinstance(record, dict):
            raise ValueError(f"{where} must be an object")
        image_id = record.get("image_id")
        assignment = assignments.get(image_id)
        if assignment is None or image_id in seen_source_images:
            raise ValueError(f"{where} has a missing, extra, or duplicate image_id")
        if assignment != (record.get("split"), record.get("group_id"), record.get("scene_type")):
            raise ValueError(f"{where} does not match split_manifest.csv")
        # These are project declarations, not proof of external rights; require approval for every training image.
        rights = record.get("rights")
        if (not isinstance(rights, dict)
                or not isinstance(rights.get("training_use"), str)
                or rights.get("training_use") != "approved"
                or not isinstance(rights.get("basis"), str)
                or rights.get("basis") not in {"owned", "permission", "license", "public_domain"}
                or not isinstance(rights.get("source_reference"), str)
                or not rights["source_reference"].strip()
                or not isinstance(rights.get("license_id"), str)
                or not isinstance(rights.get("attribution"), str)
                or (rights["basis"] == "license"
                    and (not rights["license_id"].strip() or not rights["attribution"].strip()))):
            raise ValueError(
                f"{where} lacks approved, complete per-image training-rights metadata; "
                "check training_use, basis, source_reference, and required license attribution"
            )
        surface_types = record.get("depiction_surface_types")
        counts = record.get("review_class_counts")
        image_review_status = record.get("image_review_status")
        if (not isinstance(surface_types, list)
                or any(not isinstance(value, str) or value not in REVIEWED_SURFACE_TYPES for value in surface_types)
                or len(surface_types) != len(set(surface_types))
                or not isinstance(counts, dict)
                or set(counts) != set(REVIEW_CLASS_NAMES)
                or any(type(counts.get(name)) is not int or counts[name] < 0 for name in REVIEW_CLASS_NAMES)):
            raise ValueError(f"{where} has invalid depiction surface types or review_class_counts")
        if image_review_status not in {"reviewed_empty", "reviewed_with_labels"}:
            raise ValueError(f"{where} lacks a complete v7 image_review_status")
        review_annotation_count = sum(counts.values())
        if ((image_review_status == "reviewed_empty" and review_annotation_count != 0)
                or (image_review_status == "reviewed_with_labels" and review_annotation_count == 0)):
            raise ValueError(f"{where} image_review_status conflicts with its review_class_counts")
        seen_source_images.add(image_id)
        key = (assignment[0], assignment[1])
        profile = group_profiles[key]
        profile["surface_types"].update(surface_types)
        profile["image_review_statuses"].add(image_review_status)
        for name in REVIEW_CLASS_NAMES:
            profile["review_class_counts"][name] += counts[name]
    if seen_source_images != set(assignments):
        missing = sorted(set(assignments) - seen_source_images)
        raise ValueError(f"dataset_image_sources.json omits split-manifest images: {missing[:5]}")

    if not groups_csv.is_file():
        raise FileNotFoundError(
            f"Complete the converter-generated pending source-group template and upload it to {groups_csv}"
        )
    group_metadata = {}
    with groups_csv.open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        expected_columns = ("split", "group_id", "review_status", "hard_negative_sources", "hat_type_negative_sources")
        if tuple(reader.fieldnames or ()) != expected_columns:
            raise ValueError("hard_negative_source_groups.csv has an invalid header")
        for line_number, row in enumerate(reader, 2):
            where = f"hard_negative_source_groups.csv:{line_number}"
            if None in row or any(value is None for value in row.values()):
                raise ValueError(f"{where} is malformed or has missing fields")
            split = (row.get("split") or "").strip()
            group_id = (row.get("group_id") or "").strip()
            if (split != row.get("split") or group_id != row.get("group_id")
                    or split not in {"train", "val", "test"} or not group_id):
                raise ValueError(f"{where} has invalid split/group_id metadata")
            key = (split, group_id)
            if key not in group_profiles or key in group_metadata:
                raise ValueError(f"{where} has an extra or duplicate (split, group_id)")
            if row.get("review_status") != "confirmed":
                raise ValueError(f"{where} must be explicitly marked review_status=confirmed")
            hard_sources = parse_source_list(row, "hard_negative_sources", set(HARD_NEGATIVE_SOURCE_TYPES), where)
            hat_sources = parse_source_list(row, "hat_type_negative_sources", set(HAT_TYPE_NEGATIVE_SOURCES), where)
            group_metadata[key] = {
                "review_status": "confirmed",
                "hard_negative_sources": hard_sources,
                "hat_type_negative_sources": hat_sources,
            }
    if set(group_metadata) != set(group_profiles):
        missing = sorted(set(group_profiles) - set(group_metadata))
        extra = sorted(set(group_metadata) - set(group_profiles))
        raise ValueError(f"Source-group metadata must cover every split_manifest group; missing={missing[:5]}, extra={extra[:5]}")

    surface_sources = {"poster", "printed_photo", "screen"}
    for key, profile in group_profiles.items():
        metadata = group_metadata[key]
        hard_sources = set(metadata["hard_negative_sources"])
        expected_surfaces = profile["surface_types"] & surface_sources
        expected_mixed_scene = "mixed_scene" in profile["scene_types"]
        expected_loose_hat = profile["review_class_counts"]["loose_hat"] > 0
        expected_empty_background = (
            profile["scene_types"] == {"other"}
            and not profile["surface_types"]
            and profile["image_review_statuses"] == {"reviewed_empty"}
            and sum(profile["review_class_counts"].values()) == 0
        )
        if (hard_sources & surface_sources) != expected_surfaces:
            raise ValueError(f"Group {key!r} poster/photo/screen tags do not match image-source inventory")
        if ("mixed_scene" in hard_sources) != expected_mixed_scene:
            raise ValueError(f"Group {key!r} mixed_scene tag does not match split-manifest scenes")
        if ("loose_hat" in hard_sources) != expected_loose_hat:
            raise ValueError(f"Group {key!r} loose_hat tag does not match review class-6 annotation counts")
        if ("empty_background" in hard_sources) != expected_empty_background:
            raise ValueError(f"Group {key!r} empty_background tag does not match the fully reviewed empty-group definition")
        expected_hat_sources = {
            source for source in HAT_TYPE_NEGATIVE_SOURCES
            if profile["review_class_counts"][source] > 0
        }
        if set(metadata["hat_type_negative_sources"]) != expected_hat_sources:
            raise ValueError(f"Group {key!r} traffic/cap/beanie tags do not match review annotation counts")

    digest = hashlib.sha256(groups_csv.read_bytes()).hexdigest()
    return assignments, group_metadata, digest


DATASET_GROUP_ASSIGNMENTS, HARD_NEGATIVE_SOURCE_GROUPS, HARD_NEGATIVE_SOURCE_GROUPS_SHA256 = (
    load_hard_negative_source_groups(DATASET_ROOT, HARD_NEGATIVE_SOURCE_GROUPS_CSV)
)
DATA_YAML = DATASET_ROOT / "data.yaml"
print("Dataset:", DATASET_ROOT)
print("Free Colab disk GB:", round(shutil.disk_usage(WORK_ROOT).free / 1024**3, 1))


In [ ]:
CLASS_NAMES = ["hard_hat_worn", "depiction_region"]
WORN_IDS = {0}

with DATA_YAML.open(encoding="utf-8") as stream:
    data_config = yaml.safe_load(stream)

names = data_config.get("names")
if isinstance(names, dict):
    normalized_names = [names[key] for key in sorted(names, key=lambda value: int(value))]
elif isinstance(names, list):
    normalized_names = names
else:
    raise ValueError("data.yaml must define names as a list or an integer-keyed mapping")

if normalized_names != CLASS_NAMES:
    raise ValueError(
        "Class mapping must match the hardhat-only two-class training order. "
        f"Expected {CLASS_NAMES}, found {normalized_names}"
    )

def resolve_split(value):
    path = Path(value)
    return path.resolve() if path.is_absolute() else (DATASET_ROOT / path).resolve()

split_paths = {
    split: resolve_split(data_config[split])
    for split in ("train", "val", "test")
    if split in data_config
}
if set(split_paths) != {"train", "val", "test"}:
    raise ValueError("data.yaml must define train, val, and test splits")


import math

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MINIMUM_BOX_COUNTS_BY_SPLIT = {
    "train": {"hard_hat_worn": 300, "depiction_region": 300},
    "val": {"hard_hat_worn": 1, "depiction_region": 1},
    "test": {"hard_hat_worn": 100, "depiction_region": 100},
}
MINIMUM_HARD_NEGATIVE_GROUP_COUNTS = {"train": 300, "test": 100}
MINIMUM_HAT_TYPE_NEGATIVE_GROUP_COUNTS = {"train": 300, "val": 30, "test": 100}


def split_images(split):
    configured_path = split_paths[split]
    if configured_path.is_dir():
        return sorted(
            image for image in configured_path.rglob("*")
            if image.is_file() and image.suffix.lower() in IMAGE_SUFFIXES
        )
    if not configured_path.is_file():
        raise FileNotFoundError(f"Missing {split} image path: {configured_path}")
    images = []
    for value in configured_path.read_text(encoding="utf-8-sig").splitlines():
        value = value.strip()
        if not value:
            continue
        image = Path(value)
        images.append(image.resolve() if image.is_absolute() else (configured_path.parent / image).resolve())
    return images


def label_path_for_image(image_path):
    parts = list(image_path.parts)
    image_indexes = [index for index, part in enumerate(parts) if part.lower() == "images"]
    if not image_indexes:
        raise ValueError(f"Dataset image path must contain an images/ directory: {image_path}")
    parts[image_indexes[0]] = "labels"
    return Path(*parts).with_suffix(".txt")


def compute_class_box_counts():
    counts = {
        split: {name: 0 for name in CLASS_NAMES}
        for split in ("train", "val", "test")
    }
    for split in ("train", "val", "test"):
        images = split_images(split)
        if not images:
            raise ValueError(f"The {split} split contains no images")
        image_ids = [image.stem for image in images]
        expected_ids = {
            image_id for image_id, assignment in DATASET_GROUP_ASSIGNMENTS.items()
            if assignment[0] == split
        }
        if len(image_ids) != len(set(image_ids)) or set(image_ids) != expected_ids:
            raise ValueError(f"The {split} image files do not exactly match split_manifest.csv")
        expected_label_paths = {label_path_for_image(image) for image in images}
        for image_path in images:
            label_path = label_path_for_image(image_path)
            if not label_path.is_file():
                raise FileNotFoundError(f"Missing reviewed {split} label file: {label_path}")
            for line_number, line in enumerate(label_path.read_text(encoding="utf-8-sig").splitlines(), 1):
                fields = line.split()
                if not fields:
                    continue
                if len(fields) != 5:
                    raise ValueError(f"{label_path}:{line_number} must have five YOLO values")
                class_id = int(fields[0])
                values = [float(value) for value in fields[1:]]
                if (class_id not in range(len(CLASS_NAMES))
                        or not all(math.isfinite(value) for value in values)):
                    raise ValueError(f"{label_path}:{line_number} has invalid class or coordinates")
                x_center, y_center, box_width, box_height = values
                if not (0 <= x_center <= 1 and 0 <= y_center <= 1
                        and 0 < box_width <= 1 and 0 < box_height <= 1
                        and x_center - box_width / 2 >= 0 and x_center + box_width / 2 <= 1
                        and y_center - box_height / 2 >= 0 and y_center + box_height / 2 <= 1):
                    raise ValueError(f"{label_path}:{line_number} has a non-normalized box")
                counts[split][CLASS_NAMES[class_id]] += 1
        label_directories = {path.parent for path in expected_label_paths}
        if len(label_directories) != 1:
            raise ValueError(f"The {split} labels do not share one split directory")
        label_directory = next(iter(label_directories))
        actual_label_paths = {
            label for label in label_directory.glob("*.txt") if label.is_file()
        }
        if actual_label_paths != expected_label_paths:
            raise ValueError(f"The {split} label files do not exactly match its image files")
    return counts


def count_source_groups():
    return {
        source: {
            split: sum(
                split_name == split and source in metadata["hard_negative_sources"]
                for (split_name, _group_id), metadata in HARD_NEGATIVE_SOURCE_GROUPS.items()
            )
            for split in ("train", "val", "test")
        }
        for source in HARD_NEGATIVE_SOURCE_TYPES
    }


def count_hat_type_negative_groups():
    return {
        source: {
            split: sum(
                split_name == split and source in metadata["hat_type_negative_sources"]
                for (split_name, _group_id), metadata in HARD_NEGATIVE_SOURCE_GROUPS.items()
            )
            for split in ("train", "val", "test")
        }
        for source in HAT_TYPE_NEGATIVE_SOURCES
    }


DATASET_BOX_COUNTS_BY_SPLIT = compute_class_box_counts()
DATASET_HARD_NEGATIVE_SOURCE_GROUP_COUNTS = count_source_groups()
DATASET_HAT_TYPE_NEGATIVE_GROUP_COUNTS = count_hat_type_negative_groups()
DATASET_SUFFICIENCY_ERRORS = []
for split, class_minima in MINIMUM_BOX_COUNTS_BY_SPLIT.items():
    for class_name, minimum in class_minima.items():
        if DATASET_BOX_COUNTS_BY_SPLIT[split][class_name] < minimum:
            DATASET_SUFFICIENCY_ERRORS.append(
                f"{split} {class_name} boxes < {minimum}"
            )
for source in HARD_NEGATIVE_SOURCE_TYPES:
    if DATASET_HARD_NEGATIVE_SOURCE_GROUP_COUNTS[source]["train"] < MINIMUM_HARD_NEGATIVE_GROUP_COUNTS["train"]:
        DATASET_SUFFICIENCY_ERRORS.append(f"train {source} independent groups < 300")
    if DATASET_HARD_NEGATIVE_SOURCE_GROUP_COUNTS[source]["test"] < MINIMUM_HARD_NEGATIVE_GROUP_COUNTS["test"]:
        DATASET_SUFFICIENCY_ERRORS.append(f"locked-test {source} independent groups < 100")
for source in HAT_TYPE_NEGATIVE_SOURCES:
    for split, minimum in MINIMUM_HAT_TYPE_NEGATIVE_GROUP_COUNTS.items():
        actual = DATASET_HAT_TYPE_NEGATIVE_GROUP_COUNTS[source][split]
        if actual < minimum:
            split_label = {"val": "validation", "test": "locked-test"}.get(split, split)
            DATASET_SUFFICIENCY_ERRORS.append(
                f"{split_label} {source} independent groups {actual} < {minimum}"
            )
if DATASET_SUFFICIENCY_ERRORS:
    raise ValueError("Release data-sufficiency preflight failed: " + "; ".join(DATASET_SUFFICIENCY_ERRORS))
DATASET_SUFFICIENCY_PREFLIGHT_OK = True

# The training run reads images extracted on the Colab VM, not from Drive.
data_config["path"] = str(DATASET_ROOT)
data_config["names"] = {index: name for index, name in enumerate(CLASS_NAMES)}
for split, path in split_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {split} split: {path}")
    data_config[split] = str(path)

COLAB_DATA_YAML = WORK_ROOT / "data_colab.yaml"
COLAB_DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding="utf-8")
print("Validated class order and all three split paths.")
print("Colab YAML:", COLAB_DATA_YAML)


In [ ]:
import csv
import io
import json
from collections import Counter, defaultdict
import math
from PIL import Image

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CORE_DEPICTION_SURFACE_TYPES = ("poster", "printed_photo", "screen")
ALLOWED_DEPICTION_SURFACE_TYPES = set(CORE_DEPICTION_SURFACE_TYPES) | {"other_depiction"}
REQUIRED_DEPICTION_SCENES = ("depiction_only", "mixed_scene")
ALLOWED_SCENE_TYPES = set(REQUIRED_DEPICTION_SCENES) | {"other"}
REVIEWED_SCENE_METADATA_PREFLIGHT_OK = False
REVIEWED_SCENE_METADATA_PREFLIGHT_SHA256 = None
if not globals().get("DATASET_SUFFICIENCY_PREFLIGHT_OK", False):
    raise RuntimeError("Run dataset and reviewed source-group sufficiency preflight before scene metadata validation")

def normalized_box_key(box):
    return tuple(float(value) for value in box)


def label_path_for(image_path):
    relative = image_path.relative_to(DATASET_ROOT)
    if not relative.parts or relative.parts[0] != "images":
        raise ValueError(f"Image must be inside an images/ directory: {image_path}")
    label_relative = Path("labels", *relative.parts[1:]).with_suffix(".txt")
    return DATASET_ROOT / label_relative

split_counts = {}
split_image_stems = {}
split_class_ids = {}
split_class_counts = {}
split_class_boxes = {}

for split, split_path in split_paths.items():
    image_paths = (
        sorted(path for path in split_path.rglob("*") if path.suffix.lower() in IMAGE_SUFFIXES)
        if split_path.is_dir()
        else [
            image if image.is_absolute() else (split_path.parent / image).resolve()
            for image in (
                Path(line.strip())
                for line in split_path.read_text(encoding="utf-8-sig").splitlines()
                if line.strip()
            )
        ]
    )
    if not image_paths:
        raise ValueError(f"No images found in {split} split")

    split_image_stems[split] = []
    split_class_ids[split] = {}
    split_class_counts[split] = {}
    boxes = Counter()
    for image_path in image_paths:
        if not image_path.is_absolute():
            image_path = (DATASET_ROOT / image_path).resolve()
        if not image_path.is_file():
            raise FileNotFoundError(image_path)
        split_image_stems[split].append(image_path.stem)
        with Image.open(image_path) as image:
            image.verify()

        label_path = label_path_for(image_path)
        if not label_path.is_file():
            raise FileNotFoundError(
                f"Missing label file {label_path}. Use an empty .txt file for a verified background image."
            )
        image_class_ids = set()
        image_class_counts = Counter()
        image_class_boxes = defaultdict(list)
        for line_number, line in enumerate(label_path.read_text(encoding="utf-8-sig").splitlines(), 1):
            fields = line.split()
            if len(fields) != 5:
                raise ValueError(f"{label_path}:{line_number}: expected class plus normalized xywh")
            class_id = int(fields[0])
            x_center, y_center, width, height = map(float, fields[1:])
            if class_id not in range(len(CLASS_NAMES)):
                raise ValueError(f"{label_path}:{line_number}: class ID {class_id} is outside 0–1")
            values = (x_center, y_center, width, height)
            if not all(math.isfinite(value) for value in values):
                raise ValueError(f"{label_path}:{line_number}: non-finite box coordinate")
            if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1):
                raise ValueError(f"{label_path}:{line_number}: invalid normalized box")
            if x_center - width / 2 < 0 or x_center + width / 2 > 1:
                raise ValueError(f"{label_path}:{line_number}: box extends outside the image horizontally")
            if y_center - height / 2 < 0 or y_center + height / 2 > 1:
                raise ValueError(f"{label_path}:{line_number}: box extends outside the image vertically")
            boxes[class_id] += 1
            image_class_ids.add(class_id)
            image_class_counts[class_id] += 1
            image_class_boxes[class_id].append((x_center, y_center, width, height))
        split_class_ids[split][image_path.stem] = image_class_ids
        split_class_counts[split][image_path.stem] = image_class_counts
        split_class_boxes.setdefault(split, {})[image_path.stem] = image_class_boxes

    missing = [CLASS_NAMES[index] for index in range(len(CLASS_NAMES)) if boxes[index] == 0]
    if missing and split in {"train", "val"}:
        raise ValueError(f"{split} is missing examples for classes: {missing}")
    split_counts[split] = {
        "images": len(image_paths),
        "boxes": sum(boxes.values()),
        "classes": {CLASS_NAMES[index]: boxes[index] for index in range(len(CLASS_NAMES))},
    }

# Validate the reviewed scene sidecar before either smoke training or the full run.
def parse_surface_records(raw_value, field_name, image_id, require_box=False):
    try:
        records = json.loads(raw_value)
    except json.JSONDecodeError as error:
        raise ValueError(f"{field_name} for {image_id} must be a JSON array") from error
    if not isinstance(records, list):
        raise ValueError(f"{field_name} for {image_id} must be a JSON array")

    normalized = []
    seen_surface_ids = set()
    seen_box_keys = set()
    expected_keys = {"surface_id", "surface_type", "box"} if require_box else {"surface_id", "surface_type"}
    for index, record in enumerate(records):
        if not isinstance(record, dict) or set(record) != expected_keys:
            expected = "surface_id, surface_type, and box" if require_box else "surface_id and surface_type"
            raise ValueError(f"{field_name}[{index}] for {image_id} must contain only {expected}")
        surface_id = record.get("surface_id")
        surface_type = record.get("surface_type")
        if (
            not isinstance(surface_id, str)
            or not surface_id
            or surface_id != surface_id.strip()
            or not isinstance(surface_type, str)
            or surface_type not in ALLOWED_DEPICTION_SURFACE_TYPES
        ):
            raise ValueError(
                f"{field_name}[{index}] for {image_id} has an empty ID or unsupported surface_type"
            )
        if surface_id in seen_surface_ids:
            raise ValueError(f"{field_name} for {image_id} contains duplicate surface_id {surface_id!r}")
        seen_surface_ids.add(surface_id)
        normalized_record = {"surface_id": surface_id, "surface_type": surface_type}
        if require_box:
            raw_box = record.get("box")
            if (
                not isinstance(raw_box, list)
                or len(raw_box) != 4
                or any(isinstance(value, bool) or not isinstance(value, (int, float)) for value in raw_box)
            ):
                raise ValueError(f"{field_name}[{index}] for {image_id} must include a normalized YOLO xywh box")
            box = tuple(float(value) for value in raw_box)
            x_center, y_center, width, height = box
            if (
                not all(math.isfinite(value) for value in box)
                or any(round(value, 8) != value for value in box)
                or not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1)
                or x_center - width / 2 < 0
                or x_center + width / 2 > 1
                or y_center - height / 2 < 0
                or y_center + height / 2 > 1
            ):
                raise ValueError(f"{field_name}[{index}] for {image_id} has an invalid normalized YOLO xywh box")
            box_key = normalized_box_key(box)
            if box_key in seen_box_keys:
                raise ValueError(f"{field_name} for {image_id} contains duplicate GT boxes")
            seen_box_keys.add(box_key)
            normalized_record["box"] = list(box)
        normalized.append(normalized_record)
    return normalized


def load_reviewed_scene_metadata(metadata_path, expected_image_ids):
    metadata_bytes = Path(metadata_path).read_bytes()
    metadata_text = metadata_bytes.decode("utf-8-sig")
    reader = csv.DictReader(io.StringIO(metadata_text, newline=""))
    required_columns = [
        "image_id",
        "scene_type",
        "depiction_surface_types",
        "depiction_surfaces",
        "depiction_region_surfaces",
    ]
    if (
        not reader.fieldnames
        or set(reader.fieldnames) != set(required_columns)
        or len(reader.fieldnames) != len(required_columns)
        or any(reader.fieldnames.count(column) != 1 for column in required_columns)
    ):
        raise ValueError(
            "Scene metadata columns must be image_id, scene_type, depiction_surface_types, "
            "depiction_surfaces, depiction_region_surfaces"
        )

    scene_metadata = {}
    surface_types_by_image = {}
    surfaces_by_image = {}
    region_surfaces_by_image = {}
    for row in reader:
        if None in row or any(value is None for value in row.values()):
            raise ValueError("Scene metadata contains a malformed row")
        image_id = row.get("image_id")
        scene_type = row.get("scene_type")
        raw_surface_types = row.get("depiction_surface_types")
        raw_surfaces = row.get("depiction_surfaces")
        raw_region_surfaces = row.get("depiction_region_surfaces")
        if not all(
            isinstance(value, str)
            for value in (image_id, scene_type, raw_surface_types, raw_surfaces, raw_region_surfaces)
        ):
            raise ValueError("Scene metadata values must be text")
        image_id = image_id.strip()
        scene_type = scene_type.strip()
        if not image_id or image_id in scene_metadata or scene_type not in ALLOWED_SCENE_TYPES:
            raise ValueError(f"Invalid or duplicate scene metadata row: {row}")

        try:
            surface_types = json.loads(raw_surface_types)
        except json.JSONDecodeError as error:
            raise ValueError(f"depiction_surface_types for {image_id} must be a JSON array") from error
        if (
            not isinstance(surface_types, list)
            or any(not isinstance(value, str) or value not in ALLOWED_DEPICTION_SURFACE_TYPES for value in surface_types)
            or len(surface_types) != len(set(surface_types))
        ):
            raise ValueError(
                f"depiction_surface_types for {image_id} must be a unique JSON array "
                "of allowed surface values"
            )

        surfaces = parse_surface_records(raw_surfaces, "depiction_surfaces", image_id)
        region_surfaces = parse_surface_records(
            raw_region_surfaces, "depiction_region_surfaces", image_id, require_box=True
        )
        inventory_pairs = {
            (record["surface_id"], record["surface_type"]) for record in surfaces
        }
        region_pairs = {
            (record["surface_id"], record["surface_type"]) for record in region_surfaces
        }
        inventory_types = {record["surface_type"] for record in surfaces}
        if inventory_types != set(surface_types):
            raise ValueError(
                f"depiction_surfaces for {image_id} must cover exactly the declared surface types"
            )
        if scene_type in REQUIRED_DEPICTION_SCENES:
            if not surfaces:
                raise ValueError(f"depiction_surfaces for {image_id} cannot be empty in {scene_type}")
            if region_pairs != inventory_pairs:
                raise ValueError(
                    f"Every surface instance for {image_id} must map one-to-one to a model class-1 depiction_region "
                    "with the same surface_id and surface_type"
                )
        elif surface_types or surfaces or region_surfaces:
            raise ValueError(
                f"scene_type='other' for {image_id} cannot declare depiction surfaces or regions"
            )

        scene_metadata[image_id] = scene_type
        surface_types_by_image[image_id] = surface_types
        surfaces_by_image[image_id] = surfaces
        region_surfaces_by_image[image_id] = region_surfaces

    if set(scene_metadata) != set(expected_image_ids):
        raise ValueError("Scene metadata must have exactly one row for each locked-test image")
    return scene_metadata, surface_types_by_image, surfaces_by_image, region_surfaces_by_image, metadata_bytes

test_stems = split_image_stems["test"]
test_ids = set(test_stems)
if len(test_stems) != len(test_ids):
    raise ValueError("Locked-test image stems must be unique for exact scene metadata coverage")
if not SCENE_METADATA_CSV.is_file():
    raise FileNotFoundError(
        f"Upload the reviewed test scene metadata CSV to {SCENE_METADATA_CSV} before training"
    )

early_scene_metadata, early_surface_types_by_image, early_surfaces_by_image, early_region_surfaces_by_image, metadata_bytes = load_reviewed_scene_metadata(
    SCENE_METADATA_CSV, test_ids
)
# Cross-check each locked-test group's poster/photo/screen source tags against the separate reviewed test inventory.
test_group_surface_types = defaultdict(set)
for image_id, surface_types in early_surface_types_by_image.items():
    split, group_id, _scene_type = DATASET_GROUP_ASSIGNMENTS[image_id]
    test_group_surface_types[(split, group_id)].update(surface_types)
for group_key, surface_types in test_group_surface_types.items():
    reviewed_surface_sources = set(HARD_NEGATIVE_SOURCE_GROUPS[group_key]["hard_negative_sources"]) & set(CORE_DEPICTION_SURFACE_TYPES)
    test_sidecar_surface_sources = set(surface_types) & set(CORE_DEPICTION_SURFACE_TYPES)
    if reviewed_surface_sources != test_sidecar_surface_sources:
        raise ValueError(
            f"Locked-test source-group tags for {group_key!r} do not match test_scene_metadata.csv"
        )
early_strata_counts = {
    (surface_type, scene_type): sum(
        early_scene_metadata[image_id] == scene_type
        and surface_type in early_surface_types_by_image[image_id]
        for image_id in early_scene_metadata
    )
    for surface_type in CORE_DEPICTION_SURFACE_TYPES
    for scene_type in REQUIRED_DEPICTION_SCENES
}
missing_strata = [
    f"{surface_type}/{scene_type}"
    for (surface_type, scene_type), count in early_strata_counts.items()
    if count <= 0
]
if missing_strata:
    raise ValueError(
        "Reviewed test metadata must include each core depiction surface in both "
        f"required scenes; missing: {missing_strata}"
    )
missing_surface_instance_regions = [
    image_id
    for image_id in early_scene_metadata
    if (
        split_class_counts["test"][image_id].get(1, 0) != len(early_surfaces_by_image[image_id])
        or Counter(
            normalized_box_key(record["box"])
            for record in early_region_surfaces_by_image[image_id]
        ) != Counter(
            normalized_box_key(box)
            for box in split_class_boxes["test"][image_id].get(1, [])
        )
        or len({
            normalized_box_key(box)
            for box in split_class_boxes["test"][image_id].get(1, [])
        }) != split_class_counts["test"][image_id].get(1, 0)
    )
]
if missing_surface_instance_regions:
    raise ValueError(
        "Every locked-test image must have exactly one model class-1 depiction_region label remapped from review ID 7 with matching "
        "surface_id/surface_type and normalized box per declared surface instance; "
        f"missing, duplicate, or extra mappings: {missing_surface_instance_regions[:10]}"
    )
invalid_other_depiction_labels = [
    image_id for image_id, scene_type in early_scene_metadata.items()
    if scene_type == "other"
    and split_class_ids["test"][image_id].intersection({1})
]
if invalid_other_depiction_labels:
    raise ValueError(
        f"Images marked other cannot contain depiction_region model labels: {invalid_other_depiction_labels[:10]}"
    )
print(
    "Reviewed v7 metadata covers every locked-test image and all six strata; "
    "each physical surface instance maps to exactly one matching model class-1 region box by ID, type, and normalized coordinates."
)
REVIEWED_SCENE_METADATA_PREFLIGHT_SHA256 = hashlib.sha256(metadata_bytes).hexdigest()
REVIEWED_SCENE_METADATA_PREFLIGHT_OK = True

def require_reviewed_scene_metadata_preflight():
    if not REVIEWED_SCENE_METADATA_PREFLIGHT_OK or not SCENE_METADATA_CSV.is_file():
        raise RuntimeError("Run the reviewed test scene metadata preflight before training")
    current_sha256 = hashlib.sha256(SCENE_METADATA_CSV.read_bytes()).hexdigest()
    if current_sha256 != REVIEWED_SCENE_METADATA_PREFLIGHT_SHA256:
        raise RuntimeError("Reviewed test scene metadata changed; rerun the preflight before training")

# Invalidate old release claims only after the complete dataset/metadata preflight succeeds.
for stale_path in (
    DRIVE_RUN / "deployment_release.json",
    DRIVE_RUN / "deployment_test_summary.json",
):
    stale_path.unlink(missing_ok=True)
    stale_path.with_name(stale_path.name + ".partial").unlink(missing_ok=True)

split_counts


## Kiểm tra nhanh trước khi huấn luyện

Cell kế tiếp chạy tối đa 3 epoch trên toàn bộ train split để kiểm tra việc nạp dữ liệu và bộ nhãn. Đây là run riêng. Xem loss và ảnh dự đoán trước khi chuyển sang huấn luyện chính.

Trước cell smoke, cell kiểm tra dữ liệu đã xác nhận `test_scene_metadata.csv` có đúng năm cột `image_id`, `scene_type`, `depiction_surface_types`, `depiction_surfaces`, `depiction_region_surfaces`; scene là `depiction_only`, `mixed_scene` hoặc `other`; danh sách surface là JSON array; mỗi inventory có `surface_id` duy nhất; từng review-ID-7 region record có `{surface_id, surface_type, box}` với `box` là YOLO xywh chuẩn hóa. Mapping ID/type phải một-một, và mỗi box sidecar phải khớp một model class-1 label đã remap từ review ID 7 theo bốn tọa độ canonical 8 chữ số; thiếu/sai mapping hoặc sidecar sẽ dừng trước mọi epoch. Mỗi ảnh test phải được rà soát và đủ cả sáu tổ hợp poster/ảnh in/màn hình × depiction_only/mixed_scene.

Train và validation phải có hardhat mục tiêu cùng class phụ `depiction_region`; ảnh chỉ có mũ ngoài mục tiêu phải còn nhãn rỗng làm nền âm. Tập test chỉ dùng đánh giá cuối, không dùng chọn checkpoint hoặc confidence threshold.


In [ ]:
import os
import json
import sys
if not os.environ.get("COLAB_RELEASE_TAG"):
    raise RuntimeError(
        "Google Colab managed runtime required. Do not run this training notebook on local Jupyter."
    )
import torch
import torchvision
import ultralytics
from ultralytics import YOLO

if ultralytics.__version__ != "8.4.202":
    raise RuntimeError(f"Expected Ultralytics 8.4.202, found {ultralytics.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("Switch the Colab runtime to a GPU before training")
if not globals().get("REVIEWED_SCENE_METADATA_PREFLIGHT_OK", False):
    raise RuntimeError("Run the reviewed test scene metadata preflight before smoke training")
require_reviewed_scene_metadata_preflight()
if not globals().get("DATASET_SUFFICIENCY_PREFLIGHT_OK", False):
    raise RuntimeError("Run the box-count and reviewed source-group sufficiency preflight before smoke")

def get_training_runtime_metadata():
    return {
        "python": sys.version.split()[0],
        "torch": str(torch.__version__),
        "torchvision": str(torchvision.__version__),
        "torch_cuda": str(torch.version.cuda or ""),
        "ultralytics": str(ultralytics.__version__),
    }

TRAINING_RUNTIME_METADATA = get_training_runtime_metadata()
if not all(TRAINING_RUNTIME_METADATA.values()):
    raise RuntimeError("Could not record complete cloud training runtime version metadata")

def load_persisted_training_runtime_metadata():
    runtime_path = DRIVE_RUN / "training_runtime.json"
    if not runtime_path.is_file():
        raise RuntimeError(
            "This run has no stored runtime fingerprint; do not resume or release it. "
            "Start a fresh run with a new RUN_NAME."
        )
    try:
        persisted = json.loads(runtime_path.read_text(encoding="utf-8"))
    except (OSError, UnicodeError, ValueError) as error:
        raise RuntimeError("The stored training runtime fingerprint is invalid") from error
    if (persisted != TRAINING_RUNTIME_METADATA
            or get_training_runtime_metadata() != TRAINING_RUNTIME_METADATA):
        raise RuntimeError(
            "The current Colab Python/Torch/Torchvision/CUDA/Ultralytics versions do not "
            "match the versions that started this run. Resume is blocked."
        )
    return persisted

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA runtime:", torch.version.cuda)
print("Ultralytics:", ultralytics.__version__)

smoke_model = YOLO("yolov8s.pt")
smoke_results = smoke_model.train(
    data=str(COLAB_DATA_YAML),
    epochs=3,
    imgsz=640,
    batch=16,
    seed=42,
    device=0,
    workers=2,
    project=str(LOCAL_RUNS),
    name="hat_types_smoke",
    exist_ok=False,
    plots=True,
    save_period=1,
)
print("Smoke run saved at:", LOCAL_RUNS / "hat_types_smoke")


## Huấn luyện chính và khôi phục sau khi Colab ngắt

Mặc định chạy mới từ trọng số YOLOv8s pretrained trong Ultralytics. Nếu có checkpoint YOLOv8s hai lớp đáng tin cậy của dự án trên Drive, đặt `INITIAL_WEIGHTS` đến tệp đó. Ultralytics sẽ nạp các trọng số tương thích và khởi tạo đầu phân loại theo hai lớp của hardhat-only model.

Đặt `RUN_MODE` thành `full` cho lần huấn luyện 30 epoch. Nếu runtime bị ngắt, mở lại notebook, chạy các cell chuẩn bị dữ liệu và đặt `RUN_MODE` thành `resume`. Cell sẽ lấy checkpoint `last.pt` đã đồng bộ lên Drive và tiếp tục cùng run. Không chạy mode full lên run đã tồn tại.

Tên run `RUN_NAME` được khai báo trong cell 2 để preflight dọn đúng artifact cũ; hãy đổi tên tại cell 2 khi tạo run mới.


In [ ]:
import os
if not os.environ.get("COLAB_RELEASE_TAG"):
    raise RuntimeError(
        "Google Colab managed runtime required. Do not run this training notebook on local Jupyter."
    )
if not globals().get("REVIEWED_SCENE_METADATA_PREFLIGHT_OK", False):
    raise RuntimeError("Run the reviewed test scene metadata preflight before full training")
require_reviewed_scene_metadata_preflight()
if not globals().get("DATASET_SUFFICIENCY_PREFLIGHT_OK", False):
    raise RuntimeError("Run the box-count and reviewed source-group sufficiency preflight before training")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("A managed Colab GPU runtime is required; local training is disabled.")
if get_training_runtime_metadata() != TRAINING_RUNTIME_METADATA:
    raise RuntimeError("Training runtime changed after the preflight/smoke setup")
RUN_MODE = "full"  # "full" for a new 30-epoch run, or "resume" after interruption
INITIAL_WEIGHTS = DRIVE_ROOT / "weights" / "v8s" / "best.pt"
LOCAL_RUN = LOCAL_RUNS / RUN_NAME

def copy_atomically(source, destination):
    if not source.is_file():
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".partial")
    shutil.copy2(source, temporary)
    temporary.replace(destination)

def sync_run_to_drive(trainer):
    run_dir = Path(trainer.save_dir)
    for relative in (
        "weights/last.pt",
        "weights/best.pt",
        "results.csv",
        "args.yaml",
    ):
        copy_atomically(run_dir / relative, DRIVE_RUN / relative)

if RUN_MODE not in {"full", "resume"}:
    raise ValueError("RUN_MODE must be full or resume")

if RUN_MODE == "resume":
    load_persisted_training_runtime_metadata()
    drive_last = DRIVE_RUN / "weights" / "last.pt"
    if not drive_last.is_file():
        raise FileNotFoundError(f"No cloud checkpoint to resume: {drive_last}")
    for relative in ("weights/last.pt", "weights/best.pt", "results.csv", "args.yaml"):
        source = DRIVE_RUN / relative
        if source.is_file():
            copy_atomically(source, LOCAL_RUN / relative)
    model = YOLO(str(LOCAL_RUN / "weights" / "last.pt"))
    model.add_callback("on_model_save", sync_run_to_drive)
    train_result = model.train(
        resume=True,
        data=str(COLAB_DATA_YAML),
        device=0,
    )
else:
    if DRIVE_RUN.exists() or LOCAL_RUN.exists():
        raise FileExistsError(
            "This run name already exists. Choose a new RUN_NAME for a fresh run "
            "or use RUN_MODE='resume' for an interrupted run."
        )
    TRAINING_RUNTIME_METADATA_PATH = DRIVE_RUN / "training_runtime.json"
    DRIVE_RUN.mkdir(parents=True, exist_ok=False)
    runtime_partial = TRAINING_RUNTIME_METADATA_PATH.with_name(
        TRAINING_RUNTIME_METADATA_PATH.name + ".partial"
    )
    runtime_partial.write_text(
        json.dumps(TRAINING_RUNTIME_METADATA, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    runtime_partial.replace(TRAINING_RUNTIME_METADATA_PATH)
    if INITIAL_WEIGHTS.is_file():
        start_weights = str(INITIAL_WEIGHTS)
        print("Initializing from project checkpoint:", start_weights)
    else:
        start_weights = "yolov8s.pt"
        print("Initializing from pretrained YOLOv8s weights.")

    model = YOLO(start_weights)
    model.add_callback("on_model_save", sync_run_to_drive)
    train_result = model.train(
        data=str(COLAB_DATA_YAML),
        epochs=30,
        imgsz=640,
        batch=16,
        seed=42,
        device=0,
        workers=2,
        project=str(LOCAL_RUNS),
        name=RUN_NAME,
        exist_ok=False,
        plots=True,
        save_period=1,
    )
    sync_run_to_drive(type("RunInfo", (), {"save_dir": LOCAL_RUN})())

print("Training output:", LOCAL_RUN)
print("Drive checkpoint folder:", DRIVE_RUN)


## Đánh giá cuối trên test đã khóa

Chọn checkpoint trên validation rồi chạy cell validation trước test. Cell 10 xét các group subtype-only sạch (tối thiểu 30 group cho mỗi traffic/cap/beanie), chọn threshold chung thấp nhất cao hơn confidence hardhat lớn nhất một margin `1e-6`, rồi ghi `validation_threshold_evidence.json` với hash checkpoint/dataset/metadata/validation images và labels. Cell 11 nạp và xác minh evidence này trước khi mở locked test; test summary và receipt v9 ràng buộc hash evidence. Chạy cell test một lần trên toàn bộ test. Cell tính precision/recall ứng viên detector từng lớp sau bộ lọc poster, số hộp bị suppression, số hộp `uncertain` và false candidate theo cảnh. Ứng viên zero-overlap vẫn chưa xác minh và không phải số người đã xác nhận đội mũ.

Cổng phát hành fail-closed: `datasets/test_scene_metadata.csv` phải có đúng năm cột `image_id`, `scene_type`, `depiction_surface_types`, `depiction_surfaces`, `depiction_region_surfaces`, đúng một hàng cho mỗi ảnh test; scene là `depiction_only`, `mixed_scene` hoặc `other`; hai trường surface là JSON array. Scene `other` không được có surface hoặc region type. Mỗi ảnh depiction_only/mixed_scene phải có mapping model class-1 `depiction_region` (được remap từ class 7 của JSON review) một-một cho từng physical surface ID/type đã khai báo; `depiction_region_surfaces` chứa thêm YOLO xywh chuẩn hóa trong `box`, và cell so khớp chính xác multiset với model class-1 region label thực tế, đồng thời từ chối box trùng. Sáu strata poster/ảnh in/màn hình × depiction_only/mixed_scene phải đều có ít nhất một ảnh và `depiction_surface_instance_image_count == image_count`; số false unverified hardhat candidates phải bằng 0. Trong mixed_scene, ứng viên `hard_hat_worn` chưa xác minh chỉ được ghép với ground-truth `hard_hat_worn` nếu không giao với ground-truth model class-1 `depiction_region`; mọi giao dương với vùng này đều không được ghép và được tính false cho gate. Nếu metadata thiếu hoặc không hợp lệ, notebook vẫn lưu báo cáo chẩn đoán với scene suy từ nhãn nhưng không cấp release. Khi gate fail, receipt và summary triển khai cũ bị xóa, cell dừng sau khi lưu báo cáo.

Cổng từ chối subtype cũng bắt buộc: Notebook đối chiếu metadata ảnh v4 với split manifest; một ảnh chỉ đủ điều kiện khi thuộc locked test, scene `other`, không có bất kỳ surface/region nào, có đúng một subtype traffic/cap/beanie và không có hardhat hoặc review class khác. Mỗi subtype cần ít nhất 100 `group_id` locked-test độc lập. Ở threshold đã khóa trên validation phải có 0 hộp `hard_hat_worn` và 0 ảnh có dự đoán trong từng subtype. Thiếu/sai metadata, thiếu group hoặc một dự đoán false đều chặn release.

Group subtype-only phải sạch ở mọi ảnh cùng `group_id`: không có hardhat, subtype khác hoặc surface/region; ít nhất một ảnh trong group có subtype đang xét. Tất cả ảnh trong group đủ điều kiện được chấm dự đoán; tối thiểu 100 group cho mỗi subtype và phải có 0 hardhat box/ảnh false. Chỉ khi mọi gate pass, notebook ghi `deployment_test_summary.json` và `deployment_release.json` cạnh thư mục run. Receipt v9 ghép checkpoint `best.pt` theo SHA-256, thứ tự lớp, nguồn metadata đã rà soát, các surface bắt buộc, SHA-256 archive dataset, summary test và bằng chứng threshold validation.

Mọi ảnh trong test phải có tệp nhãn; ảnh nền đã được rà xác nhận cần tệp nhãn rỗng.


In [ ]:
import csv
import hashlib
import json
import math
import os
import torch
from collections import defaultdict
from datetime import datetime, timezone

if not os.environ.get("COLAB_RELEASE_TAG"):
    raise RuntimeError("Google Colab managed runtime required; do not evaluate this checkpoint locally.")
if not torch.cuda.is_available():
    raise RuntimeError("A managed Colab GPU runtime is required for checkpoint validation.")
load_persisted_training_runtime_metadata()
if not globals().get("DATASET_SUFFICIENCY_PREFLIGHT_OK", False):
    raise RuntimeError("Run dataset and reviewed source-group sufficiency preflight before checkpoint validation")
require_reviewed_scene_metadata_preflight()

release_manifest_path = DRIVE_RUN / "deployment_release.json"
deployment_test_summary_path = DRIVE_RUN / "deployment_test_summary.json"
best_path = DRIVE_RUN / "weights" / "best.pt"
if not best_path.is_file():
    raise FileNotFoundError(f"Missing trained checkpoint: {best_path}")

best_model = YOLO(str(best_path))
actual_names = best_model.names
if isinstance(actual_names, dict):
    normalized_actual = [actual_names[index] for index in range(len(actual_names))]
else:
    normalized_actual = list(actual_names)
if normalized_actual != CLASS_NAMES:
    raise ValueError(f"Checkpoint class order differs: {normalized_actual}")

# Preserve any prior release until the required checkpoint exists and its class order is valid.
for stale_path in (release_manifest_path, deployment_test_summary_path):
    stale_path.unlink(missing_ok=True)
    stale_path.with_name(stale_path.name + ".partial").unlink(missing_ok=True)

val_metrics = best_model.val(
    data=str(COLAB_DATA_YAML),
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project=str(DRIVE_RUNS),
    name="hat_types_validation",
    exist_ok=True,
)
print("Validation mAP50-95:", float(val_metrics.box.map))

MIN_MODEL_CONFIDENCE = 0.001
MINIMUM_SUBTYPE_VALIDATION_GROUPS = MINIMUM_HAT_TYPE_NEGATIVE_GROUP_COUNTS["val"]
SUBTYPE_VALIDATION_SOURCES = ("traffic_helmet_worn", "cap_worn", "beanie_worn")
SUBTYPE_VALIDATION_EXCLUDED_CLASSES = (
    "hard_hat_worn", "bare_head", "depicted_hat", "loose_hat", "depiction_region",
    "traffic_helmet_worn", "cap_worn", "beanie_worn",
)
VALIDATION_THRESHOLD_METHOD = "minimum_threshold_above_max_hardhat_score_on_subtype_only_validation_groups_v1"
VALIDATION_THRESHOLD_MARGIN = 1e-6
VALIDATION_THRESHOLD_EVIDENCE_PATH = DRIVE_RUN / "validation_threshold_evidence.json"

source_path = DATASET_ROOT / "dataset_image_sources.json"
manifest_path = DATASET_ROOT / "split_manifest.csv"
source_bytes = source_path.read_bytes()
manifest_bytes = manifest_path.read_bytes()
source_document = json.loads(source_bytes.decode("utf-8"), object_pairs_hook=reject_duplicate_json_keys)
if (not isinstance(source_document, dict)
        or source_document.get("format") != "helmet-vision-image-sources-v4"
        or source_document.get("review_format") != "helmet-vision-review-v7"
        or not isinstance(source_document.get("images"), list)):
    raise ValueError("Validation threshold selection requires image-sources-v4 review-v7 metadata")
source_records_by_image = {}
for record in source_document["images"]:
    image_id = record.get("image_id") if isinstance(record, dict) else None
    if not isinstance(image_id, str) or image_id in source_records_by_image:
        raise ValueError("Validation image-source metadata contains a missing or duplicate image_id")
    assignment = DATASET_GROUP_ASSIGNMENTS.get(image_id)
    if assignment is None or assignment != (record.get("split"), record.get("group_id"), record.get("scene_type")):
        raise ValueError(f"Validation image-source metadata does not match split/group manifest for {image_id!r}")
    counts = record.get("review_class_counts")
    if (not isinstance(counts, dict) or set(counts) != set(REVIEW_CLASS_NAMES)
            or any(type(counts.get(name)) is not int or counts[name] < 0 for name in REVIEW_CLASS_NAMES)
            or not isinstance(record.get("depiction_surface_types"), list)
            or not isinstance(record.get("depiction_surfaces"), list)
            or not isinstance(record.get("depiction_region_surfaces"), list)):
        raise ValueError(f"Validation image-source record is incomplete for {image_id!r}")
    source_records_by_image[image_id] = record
if set(source_records_by_image) != set(DATASET_GROUP_ASSIGNMENTS):
    raise ValueError("Validation image-source metadata does not exactly cover split_manifest.csv")

validation_images = split_images("val")
validation_image_ids = [image_path.stem for image_path in validation_images]
expected_validation_ids = {
    image_id for image_id, assignment in DATASET_GROUP_ASSIGNMENTS.items() if assignment[0] == "val"
}
if len(validation_image_ids) != len(set(validation_image_ids)) or set(validation_image_ids) != expected_validation_ids:
    raise ValueError("Validation image files do not exactly match the reviewed split manifest")
validation_group_records = defaultdict(list)
for image_id in expected_validation_ids:
    validation_group_records[DATASET_GROUP_ASSIGNMENTS[image_id][1]].append(
        (image_id, source_records_by_image[image_id])
    )
validation_eligible_group_ids = {}
validation_eligible_image_ids = {}
for subtype in SUBTYPE_VALIDATION_SOURCES:
    eligible_groups = set()
    eligible_images = set()
    for group_id, members in validation_group_records.items():
        member_records = [record for _, record in members]
        group_is_pure = all(
            record.get("split") == "val"
            and record.get("scene_type") == "other"
            and record.get("depiction_surface_types") == []
            and record.get("depiction_surfaces") == []
            and record.get("depiction_region_surfaces") == []
            and all(record["review_class_counts"][name] == 0
                    for name in SUBTYPE_VALIDATION_EXCLUDED_CLASSES if name != subtype)
            for record in member_records
        )
        group_has_subtype = any(
            record["review_class_counts"][subtype] > 0 for record in member_records
        )
        if group_is_pure and group_has_subtype:
            eligible_groups.add(group_id)
            eligible_images.update(image_id for image_id, _ in members)
    if len(eligible_groups) < MINIMUM_SUBTYPE_VALIDATION_GROUPS:
        raise RuntimeError(
            f"Validation threshold cannot be selected: {subtype} has {len(eligible_groups)} "
            f"subtype-only groups; requires {MINIMUM_SUBTYPE_VALIDATION_GROUPS}"
        )
    validation_eligible_group_ids[subtype] = eligible_groups
    validation_eligible_image_ids[subtype] = eligible_images

validation_image_by_id = {image_path.stem: image_path for image_path in validation_images}
validation_source_image_ids = set().union(*validation_eligible_image_ids.values())
validation_scores_by_subtype = {subtype: defaultdict(list) for subtype in SUBTYPE_VALIDATION_SOURCES}
validation_label_digest = hashlib.sha256()
for image_path in validation_images:
    image_id = image_path.stem
    label_path = label_path_for_image(image_path)
    label_lines = [line.split() for line in label_path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
    label_counts = {class_id: 0 for class_id in range(len(CLASS_NAMES))}
    for fields in label_lines:
        if len(fields) != 5:
            raise ValueError(f"Malformed validation label row for {image_id!r}")
        label_counts[int(fields[0])] += 1
    source_counts = source_records_by_image[image_id]["review_class_counts"]
    if (label_counts[0] != source_counts["hard_hat_worn"]
            or label_counts[1] != source_counts["depiction_region"]):
        raise ValueError(f"Validation review counts do not match YOLO target labels for {image_id!r}")
    validation_label_digest.update(image_id.encode("utf-8") + b"\0")
    for path in (image_path, label_path):
        file_digest = hashlib.sha256(path.read_bytes()).digest()
        validation_label_digest.update(file_digest)

validation_images_with_scores = {subtype: defaultdict(list) for subtype in SUBTYPE_VALIDATION_SOURCES}
for image_id in sorted(validation_source_image_ids):
    image_path = validation_image_by_id[image_id]
    result = best_model.predict(
        source=str(image_path), imgsz=640, conf=MIN_MODEL_CONFIDENCE, device=0, verbose=False
    )[0]
    hard_hat_scores = [
        float(row[4]) for row in result.boxes.data.tolist()
        if int(row[5]) == 0 and math.isfinite(float(row[4]))
    ]
    for subtype in SUBTYPE_VALIDATION_SOURCES:
        if image_id in validation_eligible_image_ids[subtype]:
            validation_scores_by_subtype[subtype][image_id].extend(hard_hat_scores)
            validation_images_with_scores[subtype][image_id].extend(hard_hat_scores)

all_validation_negative_scores = [
    score for per_image in validation_scores_by_subtype.values()
    for scores in per_image.values() for score in scores
]
maximum_validation_negative_score = max(all_validation_negative_scores, default=None)
selected_confidence_threshold = (
    MIN_MODEL_CONFIDENCE
    if maximum_validation_negative_score is None
    else min(1.0, max(MIN_MODEL_CONFIDENCE, maximum_validation_negative_score + VALIDATION_THRESHOLD_MARGIN))
)
if not math.isfinite(selected_confidence_threshold):
    raise RuntimeError("Validation produced an invalid confidence threshold")

validation_group_gates = {}
for subtype in SUBTYPE_VALIDATION_SOURCES:
    box_count = sum(
        score >= selected_confidence_threshold
        for scores in validation_scores_by_subtype[subtype].values() for score in scores
    )
    images_with_prediction_count = sum(
        any(score >= selected_confidence_threshold for score in scores)
        for scores in validation_scores_by_subtype[subtype].values()
    )
    image_count = len(validation_eligible_image_ids[subtype])
    group_count = len(validation_eligible_group_ids[subtype])
    status = "pass" if box_count == 0 and images_with_prediction_count == 0 else "blocked"
    validation_group_gates[subtype] = {
        "eligible_validation_image_count": image_count,
        "eligible_validation_group_count": group_count,
        "minimum_validation_groups": MINIMUM_SUBTYPE_VALIDATION_GROUPS,
        "maximum_hard_hat_confidence_before_selected_threshold": max(
            (score for scores in validation_scores_by_subtype[subtype].values() for score in scores),
            default=None,
        ),
        "hard_hat_prediction_box_count_at_selected_threshold": box_count,
        "images_with_hard_hat_prediction_count_at_selected_threshold": images_with_prediction_count,
        "status": status,
    }
if any(gate["status"] != "pass" for gate in validation_group_gates.values()):
    raise RuntimeError("Validation threshold selection failed the subtype-only false-hardhat gate")

def sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

threshold_evidence = {
    "format": "helmet-vision-validation-threshold-v1",
    "status": "pass",
    "selection_method": VALIDATION_THRESHOLD_METHOD,
    "confidence_threshold": selected_confidence_threshold,
    "minimum_model_confidence": MIN_MODEL_CONFIDENCE,
    "minimum_validation_groups_per_subtype": MINIMUM_SUBTYPE_VALIDATION_GROUPS,
    "maximum_validation_negative_hard_hat_confidence": maximum_validation_negative_score,
    "validation_group_gates": validation_group_gates,
    "checkpoint_sha256": sha256_path(best_path),
    "dataset_archive_sha256": archive_sha256,
    "hard_negative_source_groups_sha256": HARD_NEGATIVE_SOURCE_GROUPS_SHA256,
    "image_source_metadata_sha256": hashlib.sha256(source_bytes).hexdigest(),
    "split_manifest_sha256": hashlib.sha256(manifest_bytes).hexdigest(),
    "validation_images_and_labels_sha256": validation_label_digest.hexdigest(),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
canonical_threshold_evidence = json.dumps(
    threshold_evidence, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False
).encode("utf-8")
threshold_evidence_sha256 = hashlib.sha256(canonical_threshold_evidence).hexdigest()
if VALIDATION_THRESHOLD_EVIDENCE_PATH.is_file():
    existing_evidence = json.loads(VALIDATION_THRESHOLD_EVIDENCE_PATH.read_text(encoding="utf-8"))
    old_comparable = {key: value for key, value in existing_evidence.items() if key != "created_at_utc"}
    new_comparable = {key: value for key, value in threshold_evidence.items() if key != "created_at_utc"}
    if old_comparable != new_comparable:
        if any(DRIVE_RUN.glob("locked_test_*")):
            raise RuntimeError("Validation evidence differs after locked-test evaluation began; use a fresh locked test")
        temporary_path = VALIDATION_THRESHOLD_EVIDENCE_PATH.with_name(
            VALIDATION_THRESHOLD_EVIDENCE_PATH.name + ".partial"
        )
        temporary_path.write_text(json.dumps(threshold_evidence, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        temporary_path.replace(VALIDATION_THRESHOLD_EVIDENCE_PATH)
        print("Replaced validation threshold evidence before locked test:", VALIDATION_THRESHOLD_EVIDENCE_PATH)
    else:
        threshold_evidence = existing_evidence
        canonical_threshold_evidence = json.dumps(
            threshold_evidence, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False
        ).encode("utf-8")
        threshold_evidence_sha256 = hashlib.sha256(canonical_threshold_evidence).hexdigest()
else:
    if any(DRIVE_RUN.glob("locked_test_*")):
        raise RuntimeError("Locked-test report exists but its pre-test validation threshold evidence is missing")
    temporary_path = VALIDATION_THRESHOLD_EVIDENCE_PATH.with_name(
        VALIDATION_THRESHOLD_EVIDENCE_PATH.name + ".partial"
    )
    temporary_path.write_text(json.dumps(threshold_evidence, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    temporary_path.replace(VALIDATION_THRESHOLD_EVIDENCE_PATH)
print("Validation confidence threshold:", threshold_evidence["confidence_threshold"])
print("Validation evidence:", VALIDATION_THRESHOLD_EVIDENCE_PATH)
print("Validation evidence SHA-256:", threshold_evidence_sha256)


In [ ]:
import csv
import hashlib
import io
import json
import math
import os
import torch
from collections import Counter, defaultdict
from datetime import datetime, timezone
from PIL import Image

def sha256_file(file_path):
    digest = hashlib.sha256()
    with Path(file_path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not os.environ.get("COLAB_RELEASE_TAG"):
    raise RuntimeError("Google Colab managed runtime required; do not evaluate this checkpoint locally.")
if not torch.cuda.is_available():
    raise RuntimeError("A managed Colab GPU runtime is required for locked-test evaluation.")
TRAINING_RUNTIME_METADATA = load_persisted_training_runtime_metadata()
if not globals().get("DATASET_SUFFICIENCY_PREFLIGHT_OK", False):
    raise RuntimeError("Run the box-count and reviewed source-group sufficiency preflight before locked-test evaluation")
RELEASE_MANIFEST_PATH = DRIVE_RUN / "deployment_release.json"
RELEASE_MANIFEST_TEMP = RELEASE_MANIFEST_PATH.with_name(RELEASE_MANIFEST_PATH.name + ".partial")
DEPLOYMENT_TEST_SUMMARY_PATH = DRIVE_RUN / "deployment_test_summary.json"
DEPLOYMENT_TEST_SUMMARY_TEMP = DEPLOYMENT_TEST_SUMMARY_PATH.with_name(DEPLOYMENT_TEST_SUMMARY_PATH.name + ".partial")
HAT_TYPE_NEGATIVE_REJECTION_SOURCES = ("traffic_helmet_worn", "cap_worn", "beanie_worn")
MINIMUM_HAT_TYPE_NEGATIVE_REJECTION_GROUPS = 100
MIN_MODEL_CONFIDENCE = 0.001  # Must match service/app.py so low-confidence regions still suppress poster candidates.
MIN_LOCKED_TEST_HARDHAT_GROUND_TRUTH = 100  # Locked-test box floor from the data-sufficiency plan.
VALIDATION_THRESHOLD_METHOD = "minimum_threshold_above_max_hardhat_score_on_subtype_only_validation_groups_v1"
MINIMUM_SUBTYPE_VALIDATION_GROUPS = MINIMUM_HAT_TYPE_NEGATIVE_GROUP_COUNTS["val"]
VALIDATION_THRESHOLD_EVIDENCE_PATH = DRIVE_RUN / "validation_threshold_evidence.json"
if not VALIDATION_THRESHOLD_EVIDENCE_PATH.is_file():
    raise FileNotFoundError("Run the validation-only confidence-threshold selection cell before locked-test evaluation")
validation_threshold_evidence_bytes = VALIDATION_THRESHOLD_EVIDENCE_PATH.read_bytes()
validation_threshold_evidence = json.loads(
    validation_threshold_evidence_bytes.decode("utf-8"),
    object_pairs_hook=reject_duplicate_json_keys,
)
if not isinstance(validation_threshold_evidence, dict):
    raise ValueError("Validation threshold evidence must be a JSON object")
validation_threshold_evidence_sha256 = hashlib.sha256(json.dumps(
    validation_threshold_evidence, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False
).encode("utf-8")).hexdigest()
CONFIDENCE_THRESHOLD = validation_threshold_evidence.get("confidence_threshold")
validation_created_at = validation_threshold_evidence.get("created_at_utc")
try:
    validation_created_at = datetime.fromisoformat(validation_created_at)
except (TypeError, ValueError) as error:
    raise ValueError("Validation threshold evidence has an invalid creation timestamp") from error
if (validation_threshold_evidence.get("format") != "helmet-vision-validation-threshold-v1"
        or validation_threshold_evidence.get("status") != "pass"
        or validation_threshold_evidence.get("selection_method") != VALIDATION_THRESHOLD_METHOD
        or type(CONFIDENCE_THRESHOLD) not in (int, float)
        or not math.isfinite(CONFIDENCE_THRESHOLD)
        or not MIN_MODEL_CONFIDENCE <= CONFIDENCE_THRESHOLD <= 1
        or validation_threshold_evidence.get("minimum_model_confidence") != MIN_MODEL_CONFIDENCE
        or validation_threshold_evidence.get("minimum_validation_groups_per_subtype") != MINIMUM_SUBTYPE_VALIDATION_GROUPS
        or validation_threshold_evidence.get("checkpoint_sha256") != sha256_file(best_path)
        or validation_threshold_evidence.get("dataset_archive_sha256") != archive_sha256
        or validation_threshold_evidence.get("hard_negative_source_groups_sha256") != HARD_NEGATIVE_SOURCE_GROUPS_SHA256
        or validation_threshold_evidence.get("image_source_metadata_sha256") != sha256_file(DATASET_ROOT / "dataset_image_sources.json")
        or validation_threshold_evidence.get("split_manifest_sha256") != sha256_file(DATASET_ROOT / "split_manifest.csv")
        or validation_created_at.tzinfo is None
        or validation_created_at >= datetime.now(timezone.utc)):
    raise ValueError("Validation threshold evidence does not match this checkpoint/dataset or was not recorded before locked-test evaluation")
validation_group_gates = validation_threshold_evidence.get("validation_group_gates")
if (not isinstance(validation_group_gates, dict)
        or set(validation_group_gates) != set(HAT_TYPE_NEGATIVE_REJECTION_SOURCES)):
    raise ValueError("Validation threshold evidence is missing one or more subtype-only validation gates")
for source, gate in validation_group_gates.items():
    if (not isinstance(gate, dict)
            or type(gate.get("eligible_validation_group_count")) is not int
            or type(gate.get("eligible_validation_image_count")) is not int
            or gate["eligible_validation_image_count"] < gate["eligible_validation_group_count"]
            or type(gate.get("minimum_validation_groups")) is not int
            or gate["eligible_validation_group_count"] < MINIMUM_SUBTYPE_VALIDATION_GROUPS
            or gate["minimum_validation_groups"] != MINIMUM_SUBTYPE_VALIDATION_GROUPS
            or type(gate.get("hard_hat_prediction_box_count_at_selected_threshold")) is not int
            or gate["hard_hat_prediction_box_count_at_selected_threshold"] != 0
            or type(gate.get("images_with_hard_hat_prediction_count_at_selected_threshold")) is not int
            or gate["images_with_hard_hat_prediction_count_at_selected_threshold"] != 0
            or gate.get("status") != "pass"):
        raise ValueError(f"Validation threshold evidence fails the subtype-only gate for {source}")
LOCKED_TEST_START_MARKER_PATH = DRIVE_RUN / "locked_test_start.json"
if LOCKED_TEST_START_MARKER_PATH.exists():
    raise RuntimeError("Locked test already started for this run; do not rerun it or change its threshold")
validation_images = split_images("val")
validation_image_ids = [image_path.stem for image_path in validation_images]
expected_validation_ids = {
    image_id for image_id, assignment in DATASET_GROUP_ASSIGNMENTS.items() if assignment[0] == "val"
}
if len(validation_image_ids) != len(set(validation_image_ids)) or set(validation_image_ids) != expected_validation_ids:
    raise ValueError("Validation image files changed after threshold selection")
validation_label_digest = hashlib.sha256()
for image_path in validation_images:
    validation_label_digest.update(image_path.stem.encode("utf-8") + b"\0")
    for path in (image_path, label_path_for_image(image_path)):
        validation_label_digest.update(hashlib.sha256(path.read_bytes()).digest())
if validation_label_digest.hexdigest() != validation_threshold_evidence.get("validation_images_and_labels_sha256"):
    raise ValueError("Validation images or labels changed after confidence threshold selection")
LOCKED_TEST_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()
LOCKED_TEST_START_MARKER = {
    "format": "helmet-vision-locked-test-start-v1",
    "validation_threshold_evidence_sha256": validation_threshold_evidence_sha256,
    "checkpoint_sha256": validation_threshold_evidence["checkpoint_sha256"],
    "dataset_archive_sha256": archive_sha256,
    "hard_negative_source_groups_sha256": HARD_NEGATIVE_SOURCE_GROUPS_SHA256,
    "started_at_utc": LOCKED_TEST_STARTED_AT_UTC,
}
canonical_locked_test_marker = json.dumps(
    LOCKED_TEST_START_MARKER, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False
).encode("utf-8")
LOCKED_TEST_START_MARKER_SHA256 = hashlib.sha256(canonical_locked_test_marker).hexdigest()
with LOCKED_TEST_START_MARKER_PATH.open("x", encoding="utf-8") as marker_stream:
    marker_stream.write(json.dumps(LOCKED_TEST_START_MARKER, ensure_ascii=False, indent=2) + "\n")
for stale_path in (RELEASE_MANIFEST_PATH, RELEASE_MANIFEST_TEMP, DEPLOYMENT_TEST_SUMMARY_PATH, DEPLOYMENT_TEST_SUMMARY_TEMP):
    stale_path.unlink(missing_ok=True)
current_assignments, current_source_groups, current_source_groups_sha256 = load_hard_negative_source_groups(
    DATASET_ROOT, HARD_NEGATIVE_SOURCE_GROUPS_CSV
)
if (current_assignments != DATASET_GROUP_ASSIGNMENTS
        or current_source_groups != HARD_NEGATIVE_SOURCE_GROUPS
        or current_source_groups_sha256 != HARD_NEGATIVE_SOURCE_GROUPS_SHA256):
    raise RuntimeError("Dataset split/group or reviewed hard-negative source metadata changed after preflight")

source_document = json.loads(
    (DATASET_ROOT / "dataset_image_sources.json").read_text(encoding="utf-8"),
    object_pairs_hook=reject_duplicate_json_keys,
)
if (not isinstance(source_document, dict)
        or source_document.get("format") != "helmet-vision-image-sources-v4"
        or source_document.get("review_format") != "helmet-vision-review-v7"
        or not isinstance(source_document.get("images"), list)):
    raise ValueError("Subtype rejection gates require validated image-sources-v4 review-v7 metadata")
source_records_by_image = {}
for record in source_document["images"]:
    image_id = record.get("image_id") if isinstance(record, dict) else None
    assignment = current_assignments.get(image_id)
    if (not isinstance(image_id, str) or assignment is None or image_id in source_records_by_image
            or assignment != (record.get("split"), record.get("group_id"), record.get("scene_type"))):
        raise ValueError("Image-source records are missing, duplicated, or do not match the locked split/group manifest")
    source_records_by_image[image_id] = record
CLASS_BOX_COUNTS_BY_SPLIT = compute_class_box_counts()
if CLASS_BOX_COUNTS_BY_SPLIT != DATASET_BOX_COUNTS_BY_SPLIT:
    raise RuntimeError("Dataset label box counts changed after the cloud training preflight")

MATCH_IOU_THRESHOLD = 0.50
DEPICTION_TRUTH_IDS = {1}  # model class 1 is the remapped depiction_region support class
REPORT_DIR = DRIVE_RUN / ("locked_test_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ"))

def intersection_area(first, second):
    overlap_width = max(0.0, min(first[2], second[2]) - max(first[0], second[0]))
    overlap_height = max(0.0, min(first[3], second[3]) - max(first[1], second[1]))
    return overlap_width * overlap_height

def area(box):
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])

def union_intersection_area(box, regions):
    clipped = []
    for region in regions:
        rectangle = (
            max(box[0], region[0]),
            max(box[1], region[1]),
            min(box[2], region[2]),
            min(box[3], region[3]),
        )
        if rectangle[2] > rectangle[0] and rectangle[3] > rectangle[1]:
            clipped.append(rectangle)

    x_edges = sorted({edge for rectangle in clipped for edge in (rectangle[0], rectangle[2])})
    covered_area = 0.0
    for left, right in zip(x_edges, x_edges[1:]):
        if right <= left:
            continue
        intervals = sorted(
            (rectangle[1], rectangle[3])
            for rectangle in clipped
            if rectangle[0] < right and rectangle[2] > left
        )
        covered_height = 0.0
        if intervals:
            start, end = intervals[0]
            for next_start, next_end in intervals[1:]:
                if next_start <= end:
                    end = max(end, next_end)
                else:
                    covered_height += end - start
                    start, end = next_start, next_end
            covered_height += end - start
        covered_area += (right - left) * covered_height
    return covered_area


def iou(first, second):
    overlap = intersection_area(first, second)
    union = area(first) + area(second) - overlap
    return overlap / union if union > 0 else 0.0

def classify_worn_overlap(worn_box, regions):
    worn_area = area(worn_box)
    if worn_area <= 0:
        return "uncertain", 0.0
    center_x = (worn_box[0] + worn_box[2]) / 2
    center_y = (worn_box[1] + worn_box[3]) / 2
    coverage = union_intersection_area(worn_box, regions) / worn_area
    center_inside = any(
        region[0] <= center_x <= region[2]
        and region[1] <= center_y <= region[3]
        for region in regions
    )
    if center_inside or coverage >= 0.50:
        return "suppressed", coverage
    if coverage > 0:
        return "uncertain", coverage
    return "unverified", 0.0

def match_same_class(truth, predictions):
    pairs = []
    for truth_index, (class_id, truth_box) in enumerate(truth):
        for prediction_index, prediction in enumerate(predictions):
            if prediction["class_id"] != class_id:
                continue
            overlap = iou(truth_box, prediction["box"])
            if overlap >= MATCH_IOU_THRESHOLD:
                pairs.append(
                    (-overlap, -prediction["confidence"], truth_index, prediction_index)
                )
    matched_truth = set()
    matched_predictions = set()
    for _, _, truth_index, prediction_index in sorted(pairs):
        if truth_index not in matched_truth and prediction_index not in matched_predictions:
            matched_truth.add(truth_index)
            matched_predictions.add(prediction_index)
    return matched_truth, matched_predictions

def overlaps_ground_truth_depiction(prediction, truth):
    return any(
        class_id in DEPICTION_TRUTH_IDS
        and intersection_area(prediction["box"], truth_box) > 0
        for class_id, truth_box in truth
    )

def test_label_path(image_path):
    parts = list(image_path.parts)
    image_indexes = [index for index, part in enumerate(parts) if part.lower() == "images"]
    if not image_indexes:
        raise ValueError(f"Test image path must contain an images/ directory: {image_path}")
    parts[image_indexes[0]] = "labels"
    return Path(*parts).with_suffix(".txt")

def read_yolo_truth(image_path, width, height):
    label_path = test_label_path(image_path)
    if not label_path.is_file():
        raise FileNotFoundError(
            f"Missing locked-test labels for {image_path}. "
            "Create an empty labels file for a verified background image."
        )
    truth = []
    depiction_region_normalized_boxes = []
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8-sig").splitlines(), 1):
        fields = line.split()
        if not fields:
            continue
        if len(fields) != 5:
            raise ValueError(f"{label_path}:{line_number} must have five YOLO values")
        class_id = int(fields[0])
        x_center, y_center, box_width, box_height = map(float, fields[1:])
        coordinates = (x_center, y_center, box_width, box_height)
        if class_id not in range(len(CLASS_NAMES)) or not all(map(math.isfinite, coordinates)):
            raise ValueError(f"{label_path}:{line_number} has invalid values")
        if not (
            0 <= x_center <= 1
            and 0 <= y_center <= 1
            and 0 < box_width <= 1
            and 0 < box_height <= 1
            and x_center - box_width / 2 >= 0
            and x_center + box_width / 2 <= 1
            and y_center - box_height / 2 >= 0
            and y_center + box_height / 2 <= 1
        ):
            raise ValueError(f"{label_path}:{line_number} has an invalid box")
        truth.append((
            class_id,
            (
                (x_center - box_width / 2) * width,
                (y_center - box_height / 2) * height,
                (x_center + box_width / 2) * width,
                (y_center + box_height / 2) * height,
            ),
        ))
        if class_id == 1:
            depiction_region_normalized_boxes.append((x_center, y_center, box_width, box_height))
    depiction_region_boxes_unique = (
        len({normalized_box_key(box) for box in depiction_region_normalized_boxes})
        == len(depiction_region_normalized_boxes)
    )
    return truth, depiction_region_normalized_boxes, depiction_region_boxes_unique

test_root = split_paths["test"]
if test_root.is_dir():
    test_images = sorted(
        image for image in test_root.rglob("*")
        if image.is_file() and image.suffix.lower() in IMAGE_SUFFIXES
    )
else:
    test_images = []
    for line in test_root.read_text(encoding="utf-8-sig").splitlines():
        value = line.strip()
        if value:
            image = Path(value)
            test_images.append(image if image.is_absolute() else (test_root.parent / image).resolve())
if not test_images:
    raise ValueError("The locked test split contains no images")
test_image_ids = [image_path.stem for image_path in test_images]
expected_locked_test_ids = {
    image_id for image_id, assignment in current_assignments.items() if assignment[0] == "test"
}
if len(test_image_ids) != len(set(test_image_ids)) or set(test_image_ids) != expected_locked_test_ids:
    raise ValueError("Locked-test image files do not exactly match reviewed per-image source metadata")
locked_test_source_records = {
    image_id: source_records_by_image[image_id] for image_id in expected_locked_test_ids
}

SUBTYPE_ONLY_EXCLUDED_CLASSES = (
    "hard_hat_worn", "bare_head", "depicted_hat", "loose_hat", "depiction_region",
    "traffic_helmet_worn", "cap_worn", "beanie_worn",
)
def is_subtype_only_test_record(record, subtype, require_subtype_present=True):
    counts = record.get("review_class_counts", {})
    return (
        record.get("split") == "test"
        and record.get("scene_type") == "other"
        and record.get("depiction_surface_types") == []
        and record.get("depiction_surfaces") == []
        and record.get("depiction_region_surfaces") == []
        and isinstance(counts, dict)
        and set(counts) == set(REVIEW_CLASS_NAMES)
        and type(counts.get(subtype)) is int
        and (not require_subtype_present or counts[subtype] > 0)
        and all(counts.get(name) == 0 for name in SUBTYPE_ONLY_EXCLUDED_CLASSES if name != subtype)
    )

test_group_records = defaultdict(list)
for image_id, record in locked_test_source_records.items():
    test_group_records[record["group_id"]].append((image_id, record))
hat_type_negative_eligible_group_ids = {}
hat_type_negative_eligible_image_ids = {}
for subtype in HAT_TYPE_NEGATIVE_SOURCES:
    eligible_groups = set()
    eligible_images = set()
    for group_id, members in test_group_records.items():
        member_records = [record for _, record in members]
        group_is_pure = all(
            is_subtype_only_test_record(record, subtype, require_subtype_present=False)
            for record in member_records
        )
        group_has_subtype = any(
            record["review_class_counts"][subtype] > 0 for record in member_records
        )
        if group_is_pure and group_has_subtype:
            eligible_groups.add(group_id)
            eligible_images.update(image_id for image_id, _ in members)
    hat_type_negative_eligible_group_ids[subtype] = eligible_groups
    hat_type_negative_eligible_image_ids[subtype] = eligible_images

checkpoint_sha256 = sha256_file(best_path)
scene_metadata = {}
depiction_surface_types_by_image = {}
depiction_surfaces_by_image = {}
depiction_region_surfaces_by_image = {}
scene_metadata_source = "inferred_from_test_labels"
scene_metadata_validation_error = "Reviewed test scene metadata is missing."
reviewed_metadata_sha256 = None
test_stems = [image.stem for image in test_images]
test_ids = set(test_stems)

try:
    (
        scene_metadata,
        depiction_surface_types_by_image,
        depiction_surfaces_by_image,
        depiction_region_surfaces_by_image,
        metadata_bytes,
    ) = load_reviewed_scene_metadata(SCENE_METADATA_CSV, test_ids)
    for image_id in test_ids:
        source_record = locked_test_source_records[image_id]
        if (source_record.get("scene_type") != scene_metadata[image_id]
                or not isinstance(source_record.get("depiction_surface_types"), list)
                or len(source_record["depiction_surface_types"]) != len(set(source_record["depiction_surface_types"]))
                or set(source_record["depiction_surface_types"]) != set(depiction_surface_types_by_image[image_id])):
            raise ValueError(f"Image-source scene/surface types disagree with reviewed CSV for {image_id!r}")
        source_surfaces = source_record.get("depiction_surfaces")
        source_regions = source_record.get("depiction_region_surfaces")
        if not isinstance(source_surfaces, list) or not isinstance(source_regions, list):
            raise ValueError(f"Image-source surface inventory is missing for {image_id!r}")
        source_inventory = Counter(
            (record.get("surface_id"), record.get("surface_type"))
            for record in source_surfaces if isinstance(record, dict)
        )
        csv_inventory = Counter(
            (record["surface_id"], record["surface_type"])
            for record in depiction_surfaces_by_image[image_id]
        )
        source_region_inventory = Counter(
            (record.get("surface_id"), record.get("surface_type"), normalized_box_key(record.get("box", [])))
            for record in source_regions if isinstance(record, dict)
        )
        csv_region_inventory = Counter(
            (record["surface_id"], record["surface_type"], normalized_box_key(record["box"]))
            for record in depiction_region_surfaces_by_image[image_id]
        )
        if (len(source_inventory) != len(source_surfaces)
                or len(source_region_inventory) != len(source_regions)
                or source_inventory != csv_inventory
                or source_region_inventory != csv_region_inventory):
            raise ValueError(f"Image-source surface inventories/regions disagree with reviewed CSV for {image_id!r}")
    scene_metadata_source = "reviewed_metadata_csv"
    scene_metadata_validation_error = None
    reviewed_metadata_sha256 = hashlib.sha256(metadata_bytes).hexdigest()
except (OSError, UnicodeError, csv.Error, ValueError) as error:
    scene_metadata = {}
    depiction_surface_types_by_image = {}
    depiction_surfaces_by_image = {}
    depiction_region_surfaces_by_image = {}
    scene_metadata_source = "inferred_from_test_labels"
    scene_metadata_validation_error = f"{type(error).__name__}: {error}"
    print(
        "Reviewed scene metadata is missing or invalid. The report will infer scene "
        "groups from test annotations, but the deployment release gate will fail."
    )

overall = defaultdict(Counter)
by_scene = defaultdict(lambda: defaultdict(Counter))
image_rows = []
prediction_rows = []
hat_type_rejection_observations = {
    source: {
        "images": set(),
        "groups": set(hat_type_negative_eligible_group_ids[source]),
        "prediction_boxes": 0,
        "images_with_prediction": set(),
    }
    for source in HAT_TYPE_NEGATIVE_REJECTION_SOURCES
}

for image_index, image_path in enumerate(test_images, start=1):
    with Image.open(image_path) as source:
        image_width, image_height = source.size
    truth, depiction_region_normalized_boxes, depiction_region_boxes_unique = read_yolo_truth(image_path, image_width, image_height)
    truth_classes = [class_id for class_id, _ in truth]
    source_review_counts = locked_test_source_records[image_path.stem]["review_class_counts"]
    if (sum(class_id == 0 for class_id in truth_classes) != source_review_counts["hard_hat_worn"]
            or sum(class_id == 1 for class_id in truth_classes) != source_review_counts["depiction_region"]):
        raise ValueError(f"Review source class counts do not match YOLO target labels for {image_path.stem!r}")
    image_surface_types = depiction_surface_types_by_image.get(image_path.stem, [])
    image_surfaces = depiction_surfaces_by_image.get(image_path.stem, [])
    image_region_surfaces = depiction_region_surfaces_by_image.get(image_path.stem, [])
    inventory_pairs = {
        (record["surface_id"], record["surface_type"]) for record in image_surfaces
    }
    region_pairs = {
        (record["surface_id"], record["surface_type"]) for record in image_region_surfaces
    }
    ground_truth_depiction_region_count = sum(class_id == 1 for class_id, _ in truth)
    region_boxes_match_ground_truth = (
        depiction_region_boxes_unique
        and Counter(normalized_box_key(record["box"]) for record in image_region_surfaces)
        == Counter(normalized_box_key(box) for box in depiction_region_normalized_boxes)
    )
    surface_instance_mapping_complete = (
        inventory_pairs == region_pairs
        and ground_truth_depiction_region_count == len(image_surfaces)
        and region_boxes_match_ground_truth
    )

    if scene_metadata:
        scene_type = scene_metadata[image_path.stem]
        # Preserve this locked-test row in the diagnostic report, but its incomplete
        # mapping prevents its per-surface instance gate from passing.
        if scene_type == "other" and any(class_id in DEPICTION_TRUTH_IDS for class_id in truth_classes):
            raise ValueError(
                f"Locked-test image {image_path.stem} is marked other but contains depiction labels"
            )
    else:
        has_region = 1 in truth_classes
        has_real_head = any(class_id in WORN_IDS for class_id in truth_classes)
        scene_type = (
            "mixed_scene" if has_region and has_real_head
            else "depiction_only" if has_region
            else "other"
        )

    prediction_result = best_model.predict(
        source=str(image_path),
        imgsz=640,
        conf=MIN_MODEL_CONFIDENCE,
        device=0,
        verbose=False,
    )[0]
    predictions = []
    for row in prediction_result.boxes.data.tolist():
        class_id = int(row[5])
        if class_id not in range(len(CLASS_NAMES)):
            raise ValueError(f"Checkpoint emitted unexpected class ID {class_id}")
        confidence = float(row[4])
        if class_id in WORN_IDS and confidence < CONFIDENCE_THRESHOLD:
            continue
        if class_id == 1 and confidence < MIN_MODEL_CONFIDENCE:
            continue
        predictions.append({
            "class_id": class_id,
            "class_name": CLASS_NAMES[class_id],
            "box": tuple(float(value) for value in row[:4]),
            "confidence": confidence,
            "filter_status": "regular",
        })

    for source in HAT_TYPE_NEGATIVE_REJECTION_SOURCES:
        if image_path.stem in hat_type_negative_eligible_image_ids[source]:
            observation = hat_type_rejection_observations[source]
            observation["images"].add(image_path.stem)
            hardhat_prediction_count = sum(prediction["class_id"] == 0 for prediction in predictions)
            observation["prediction_boxes"] += hardhat_prediction_count
            if hardhat_prediction_count:
                observation["images_with_prediction"].add(image_path.stem)

    depiction_regions = [
        item["box"] for item in predictions if item["class_id"] == 1
    ]
    for prediction in predictions:
        if prediction["class_id"] in WORN_IDS:
            status, coverage = classify_worn_overlap(
                prediction["box"], depiction_regions
            )
            prediction["filter_status"] = status
            prediction["depiction_overlap_fraction"] = coverage

    scene_counts = by_scene[scene_type]
    scene_counts["images"] += 1
    scene_counts["ground_truth_worn"] += sum(
        class_id in WORN_IDS for class_id in truth_classes
    )

    for prediction in predictions:
        if prediction["class_id"] in WORN_IDS:
            scene_counts[f"{prediction['filter_status']}_worn"] += 1

    for class_id in range(len(CLASS_NAMES)):
        class_truth = [(cls, box) for cls, box in truth if cls == class_id]
        class_predictions = [
            prediction for prediction in predictions
            if prediction["class_id"] == class_id
            and (
                class_id not in WORN_IDS
                or prediction["filter_status"] == "unverified"
            )
        ]
        matchable_prediction_indices = [
            index
            for index, prediction in enumerate(class_predictions)
            if class_id not in WORN_IDS
            or not overlaps_ground_truth_depiction(prediction, truth)
        ]
        matchable_predictions = [
            class_predictions[index] for index in matchable_prediction_indices
        ]
        matched_truth, matched_matchable_predictions = match_same_class(
            class_truth, matchable_predictions
        )
        matched_predictions = {
            matchable_prediction_indices[index]
            for index in matched_matchable_predictions
        }
        counts = overall[class_id]
        scene_class_counts = scene_counts[class_id]
        counts["TP"] += len(matched_truth)
        counts["FP"] += len(class_predictions) - len(matched_predictions)
        counts["FN"] += len(class_truth) - len(matched_truth)
        scene_class_counts["TP"] += len(matched_truth)
        scene_class_counts["FP"] += len(class_predictions) - len(matched_predictions)
        scene_class_counts["FN"] += len(class_truth) - len(matched_truth)

        for index, prediction in enumerate(class_predictions):
            if class_id in WORN_IDS and prediction["filter_status"] != "unverified":
                continue
            prediction["metric_status"] = (
                "TP" if index in matched_predictions else "FP"
            )

    unverified_worn = [
        prediction for prediction in predictions
        if prediction["class_id"] in WORN_IDS
        and prediction["filter_status"] == "unverified"
    ]
    matchable_unverified_worn = [
        prediction
        for prediction in unverified_worn
        if not overlaps_ground_truth_depiction(prediction, truth)
    ]
    matched_worn_truth, matched_unverified_worn = match_same_class(
        [(class_id, box) for class_id, box in truth if class_id in WORN_IDS],
        matchable_unverified_worn,
    )
    false_unverified_worn = len(unverified_worn) - len(matched_unverified_worn)
    scene_counts["unverified_worn_candidates"] += len(unverified_worn)
    scene_counts["matched_worn_candidates"] += len(matched_unverified_worn)
    image_false_unverified_worn_candidates = (
        len(unverified_worn) if scene_type == "depiction_only" else false_unverified_worn
    )
    scene_counts["false_unverified_worn_candidates"] += image_false_unverified_worn_candidates
    image_rows.append({
        "image_id": image_path.stem,
        "scene_type": scene_type,
        "depiction_surface_types": json.dumps(image_surface_types, ensure_ascii=False),
        "depiction_surfaces": json.dumps(image_surfaces, ensure_ascii=False),
        "depiction_region_surfaces": json.dumps(image_region_surfaces, ensure_ascii=False),
        "ground_truth_worn": sum(class_id in WORN_IDS for class_id in truth_classes),
        "ground_truth_depiction_region_count": ground_truth_depiction_region_count,
        "ground_truth_surface_instance_mapping_complete": surface_instance_mapping_complete,
        "ground_truth_surface_instance_boxes_match": region_boxes_match_ground_truth,
        "ground_truth_depiction_region_boxes_unique": depiction_region_boxes_unique,
        "unverified_worn_candidates": len(unverified_worn),
        "false_unverified_worn_candidates": image_false_unverified_worn_candidates,
        "suppressed_worn": sum(
            prediction["filter_status"] == "suppressed"
            for prediction in predictions if prediction["class_id"] in WORN_IDS
        ),
        "uncertain_worn": sum(
            prediction["filter_status"] == "uncertain"
            for prediction in predictions if prediction["class_id"] in WORN_IDS
        ),
    })
    for prediction in predictions:
        prediction_rows.append({
            "image_id": image_path.stem,
            "scene_type": scene_type,
            "depiction_surface_types": json.dumps(image_surface_types, ensure_ascii=False),
            "depiction_surfaces": json.dumps(image_surfaces, ensure_ascii=False),
            "class_name": prediction["class_name"],
            "confidence": prediction["confidence"],
            "filter_status": prediction["filter_status"],
            "overlaps_ground_truth_depiction": (
                overlaps_ground_truth_depiction(prediction, truth)
                if prediction["class_id"] in WORN_IDS
                else False
            ),
            "metric_status": prediction.get("metric_status", ""),
            "depiction_overlap_fraction": prediction.get("depiction_overlap_fraction", ""),
        })

    if image_index % 100 == 0 or image_index == len(test_images):
        print(f"Evaluated {image_index}/{len(test_images)} locked-test images", flush=True)

class_metrics = []
for class_id, class_name in enumerate(CLASS_NAMES):
    counts = overall[class_id]
    precision = counts["TP"] / (counts["TP"] + counts["FP"]) if counts["TP"] + counts["FP"] else ""
    recall = counts["TP"] / (counts["TP"] + counts["FN"]) if counts["TP"] + counts["FN"] else ""
    class_metrics.append({
        "class_id": class_id,
        "class_name": class_name,
        "TP": counts["TP"],
        "FP": counts["FP"],
        "FN": counts["FN"],
        "precision": precision,
        "recall": recall,
    })

scene_metrics = []
for scene_type in sorted(by_scene):
    scene_counts = by_scene[scene_type]
    for class_id, class_name in enumerate(CLASS_NAMES):
        counts = scene_counts[class_id]
        precision = counts["TP"] / (counts["TP"] + counts["FP"]) if counts["TP"] + counts["FP"] else ""
        recall = counts["TP"] / (counts["TP"] + counts["FN"]) if counts["TP"] + counts["FN"] else ""
        scene_metrics.append({
            "scene_type": scene_type,
            "class_name": class_name,
            "TP": counts["TP"],
            "FP": counts["FP"],
            "FN": counts["FN"],
            "precision": precision,
            "recall": recall,
            "suppressed_worn": scene_counts["suppressed_worn"],
            "uncertain_worn": scene_counts["uncertain_worn"],
            "false_unverified_worn_candidates": scene_counts["false_unverified_worn_candidates"],
        })

depiction_surface_gate_counts = {
    surface_type: {
        scene_type: {
            "image_count": 0,
            "depiction_surface_instance_image_count": 0,
            "false_unverified_worn_candidates": 0,
        }
        for scene_type in REQUIRED_DEPICTION_SCENES
    }
    for surface_type in CORE_DEPICTION_SURFACE_TYPES
}
for row in image_rows:
    scene_type = row["scene_type"]
    if scene_type not in REQUIRED_DEPICTION_SCENES:
        continue
    surface_types = json.loads(row["depiction_surface_types"])
    for surface_type in CORE_DEPICTION_SURFACE_TYPES:
        if surface_type in surface_types:
            gate_counts = depiction_surface_gate_counts[surface_type][scene_type]
            gate_counts["image_count"] += 1
            if row["ground_truth_surface_instance_mapping_complete"]:
                gate_counts["depiction_surface_instance_image_count"] += 1
            gate_counts["false_unverified_worn_candidates"] += row[
                "false_unverified_worn_candidates"
            ]

depiction_surface_test_gates = {}
for surface_type in CORE_DEPICTION_SURFACE_TYPES:
    depiction_surface_test_gates[surface_type] = {}
    for scene_type in REQUIRED_DEPICTION_SCENES:
        gate_counts = depiction_surface_gate_counts[surface_type][scene_type]
        if scene_metadata_source != "reviewed_metadata_csv":
            status = "not_evaluated_missing_reviewed_metadata"
        elif gate_counts["image_count"] <= 0:
            status = "not_evaluated_missing_required_stratum"
        elif gate_counts["depiction_surface_instance_image_count"] != gate_counts["image_count"]:
            status = "blocked_missing_ground_truth_depiction_region"
        elif gate_counts["false_unverified_worn_candidates"] == 0:
            status = "pass_reviewed_metadata"
        else:
            status = "fail_false_unverified_worn_candidates"
        depiction_surface_test_gates[surface_type][scene_type] = {
            "image_count": gate_counts["image_count"],
            "depiction_surface_instance_image_count": gate_counts["depiction_surface_instance_image_count"],
            "false_unverified_worn_candidates": gate_counts[
                "false_unverified_worn_candidates"
            ],
            "status": status,
        }

all_depiction_surface_gates_pass = all(
    depiction_surface_test_gates[surface_type][scene_type]["status"]
    == "pass_reviewed_metadata"
    for surface_type in CORE_DEPICTION_SURFACE_TYPES
    for scene_type in REQUIRED_DEPICTION_SCENES
)
depiction_surface_gate_status = (
    "pass_reviewed_metadata" if all_depiction_surface_gates_pass else "blocked"
)
hard_negative_source_group_gates = {}
for source in HARD_NEGATIVE_SOURCE_TYPES:
    train_group_count = DATASET_HARD_NEGATIVE_SOURCE_GROUP_COUNTS[source]["train"]
    locked_test_group_count = DATASET_HARD_NEGATIVE_SOURCE_GROUP_COUNTS[source]["test"]
    hard_negative_source_group_gates[source] = {
        "train_group_count": train_group_count,
        "locked_test_group_count": locked_test_group_count,
        "minimum_train_groups": MINIMUM_HARD_NEGATIVE_GROUP_COUNTS["train"],
        "minimum_locked_test_groups": MINIMUM_HARD_NEGATIVE_GROUP_COUNTS["test"],
        "status": (
            "pass"
            if train_group_count >= MINIMUM_HARD_NEGATIVE_GROUP_COUNTS["train"]
            and locked_test_group_count >= MINIMUM_HARD_NEGATIVE_GROUP_COUNTS["test"]
            else "blocked"
        ),
    }
hat_type_negative_group_counts = count_hat_type_negative_groups()
hat_type_negative_group_gates = {}
for source in HAT_TYPE_NEGATIVE_SOURCES:
    counts = hat_type_negative_group_counts[source]
    hat_type_negative_group_gates[source] = {
        "group_count_by_split": counts,
        "minimum_group_count_by_split": MINIMUM_HAT_TYPE_NEGATIVE_GROUP_COUNTS,
        "status": (
            "pass"
            if all(counts[split] >= minimum for split, minimum in MINIMUM_HAT_TYPE_NEGATIVE_GROUP_COUNTS.items())
            else "blocked"
        ),
    }
data_sufficiency_gate_reasons = []
for split, class_minima in MINIMUM_BOX_COUNTS_BY_SPLIT.items():
    for class_name, minimum in class_minima.items():
        if CLASS_BOX_COUNTS_BY_SPLIT[split][class_name] < minimum:
            data_sufficiency_gate_reasons.append(f"{split}_{class_name}_boxes_below_{minimum}")
for source, gate in hard_negative_source_group_gates.items():
    if gate["status"] != "pass":
        data_sufficiency_gate_reasons.append(f"{source}_independent_group_floor_not_met")
for source, gate in hat_type_negative_group_gates.items():
    if gate["status"] != "pass":
        for split, minimum in MINIMUM_HAT_TYPE_NEGATIVE_GROUP_COUNTS.items():
            actual = gate["group_count_by_split"][split]
            if actual < minimum:
                data_sufficiency_gate_reasons.append(
                    f"{source}_{split}_independent_groups_{actual}_below_{minimum}"
                )
data_sufficiency_gate_status = "pass" if not data_sufficiency_gate_reasons else "blocked"

hat_type_negative_rejection_gates = {}
for source in HAT_TYPE_NEGATIVE_REJECTION_SOURCES:
    observation = hat_type_rejection_observations[source]
    eligible_group_count = len(observation["groups"])
    gate_status = (
        "pass"
        if eligible_group_count >= MINIMUM_HAT_TYPE_NEGATIVE_REJECTION_GROUPS
        and observation["prediction_boxes"] == 0
        and not observation["images_with_prediction"]
        else "blocked"
    )
    hat_type_negative_rejection_gates[source] = {
        "eligible_locked_test_image_count": len(observation["images"]),
        "eligible_locked_test_group_count": eligible_group_count,
        "minimum_locked_test_groups": MINIMUM_HAT_TYPE_NEGATIVE_REJECTION_GROUPS,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "hard_hat_prediction_box_count": observation["prediction_boxes"],
        "images_with_hard_hat_prediction_count": len(observation["images_with_prediction"]),
        "status": gate_status,
    }
hat_type_negative_rejection_gate_status = (
    "pass"
    if all(gate["status"] == "pass" for gate in hat_type_negative_rejection_gates.values())
    else "blocked"
)

deployment_gate_reasons = list(data_sufficiency_gate_reasons)
for source, gate in hat_type_negative_rejection_gates.items():
    if gate["eligible_locked_test_group_count"] < MINIMUM_HAT_TYPE_NEGATIVE_REJECTION_GROUPS:
        deployment_gate_reasons.append(
            f"{source}_subtype_only_locked_test_groups_{gate['eligible_locked_test_group_count']}_below_{MINIMUM_HAT_TYPE_NEGATIVE_REJECTION_GROUPS}"
        )
    if gate["hard_hat_prediction_box_count"] or gate["images_with_hard_hat_prediction_count"]:
        deployment_gate_reasons.append(f"{source}_false_hardhat_prediction_nonzero")
if scene_metadata_source != "reviewed_metadata_csv":
    deployment_gate_reasons.append("reviewed_test_scene_metadata_missing_or_invalid")
locked_test_hardhat_ground_truth_count = sum(
    int(row["ground_truth_worn"]) for row in image_rows
)
hardhat_ground_truth_gate_status = (
    "pass"
    if locked_test_hardhat_ground_truth_count >= MIN_LOCKED_TEST_HARDHAT_GROUND_TRUTH
    else "blocked"
)
if hardhat_ground_truth_gate_status != "pass":
    deployment_gate_reasons.append("locked_test_hardhat_ground_truth_below_minimum")
for surface_type in CORE_DEPICTION_SURFACE_TYPES:
    for scene_type in REQUIRED_DEPICTION_SCENES:
        gate = depiction_surface_test_gates[surface_type][scene_type]
        if gate["image_count"] <= 0:
            deployment_gate_reasons.append(
                f"{surface_type}_{scene_type}_stratum_missing"
            )
        if gate["depiction_surface_instance_image_count"] != gate["image_count"]:
            deployment_gate_reasons.append(
                f"{surface_type}_{scene_type}_ground_truth_depiction_region_incomplete"
            )
        if gate["false_unverified_worn_candidates"] != 0:
            deployment_gate_reasons.append(
                f"{surface_type}_{scene_type}_false_unverified_worn_candidates_nonzero"
            )
deployment_gate_status = (
    "approved"
    if all_depiction_surface_gates_pass
    and hardhat_ground_truth_gate_status == "pass"
    and data_sufficiency_gate_status == "pass"
    and hat_type_negative_rejection_gate_status == "pass"
    else "blocked"
)

if REPORT_DIR.exists():
    raise FileExistsError(f"Locked-test report directory already exists: {REPORT_DIR}")
REPORT_DIR.mkdir(parents=True)

def write_report_csv(destination, rows):
    if not rows:
        return
    with destination.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

write_report_csv(REPORT_DIR / "per_class_metrics.csv", class_metrics)
write_report_csv(REPORT_DIR / "per_scene_class_metrics.csv", scene_metrics)
write_report_csv(REPORT_DIR / "per_image_scene_metrics.csv", image_rows)
write_report_csv(REPORT_DIR / "predictions_and_filter_status.csv", prediction_rows)

summary = {
    "status": "complete",
    "locked_test_images": len(test_images),
    "confidence_threshold_frozen_before_test": CONFIDENCE_THRESHOLD,
    "validation_threshold_evidence": validation_threshold_evidence,
    "validation_threshold_evidence_sha256": validation_threshold_evidence_sha256,
    "locked_test_start_marker": LOCKED_TEST_START_MARKER,
    "locked_test_start_marker_sha256": LOCKED_TEST_START_MARKER_SHA256,
    "locked_test_started_at_utc": LOCKED_TEST_STARTED_AT_UTC,
    "minimum_model_confidence": MIN_MODEL_CONFIDENCE,
    "locked_test_hardhat_ground_truth_count": locked_test_hardhat_ground_truth_count,
    "minimum_locked_test_hardhat_ground_truth": MIN_LOCKED_TEST_HARDHAT_GROUND_TRUTH,
    "hardhat_ground_truth_gate_status": hardhat_ground_truth_gate_status,
    "class_box_counts_by_split": CLASS_BOX_COUNTS_BY_SPLIT,
    "hard_negative_source_groups_sha256": current_source_groups_sha256,
    "hard_negative_source_group_gates": hard_negative_source_group_gates,
    "hat_type_negative_group_counts": hat_type_negative_group_counts,
    "minimum_hat_type_negative_group_counts": MINIMUM_HAT_TYPE_NEGATIVE_GROUP_COUNTS,
    "hat_type_negative_group_gates": hat_type_negative_group_gates,
    "hat_type_negative_rejection_gates": hat_type_negative_rejection_gates,
    "hat_type_negative_rejection_gate_status": hat_type_negative_rejection_gate_status,
    "data_sufficiency_gate_status": data_sufficiency_gate_status,
    "data_sufficiency_gate_reasons": data_sufficiency_gate_reasons,
    "matching_iou_threshold": MATCH_IOU_THRESHOLD,
    "checkpoint": str(best_path),
    "checkpoint_sha256": checkpoint_sha256,
    "training_runtime": TRAINING_RUNTIME_METADATA,
    "dataset_archive_sha256": archive_sha256,
    "scene_metadata_source": scene_metadata_source,
    "scene_metadata_evidence": {
        "path": str(SCENE_METADATA_CSV),
        "sha256": reviewed_metadata_sha256,
        "validation_error": scene_metadata_validation_error,
    },
    "deployment_gate_status": deployment_gate_status,
    "depiction_surface_gate_status": depiction_surface_gate_status,
    "required_depiction_surface_types": list(CORE_DEPICTION_SURFACE_TYPES),
    "required_depiction_scenes": list(REQUIRED_DEPICTION_SCENES),
    "depiction_surface_test_gates": depiction_surface_test_gates,
    "deployment_gate_reasons": deployment_gate_reasons,
    "wearer_filter_policy": (
        "Suppress a model class-0 hard_hat_worn box if its center is inside any detected depiction_region "
        "or at least 50 percent of wearer-box area is covered by the union of regions. "
        "Smaller positive union overlap is uncertain and never counted. Zero overlap "
        "remains an unverified candidate."
    ),
    "mixed_scene_false_candidate_matching_policy": (
        "An unverified model class-0 hard_hat_worn prediction that has any positive intersection with a "
        "ground-truth model class-1 depiction_region box is excluded from real-wearer "
        "matching and remains false for the locked-test gate."
    ),
    "uncertain_worn_predictions": sum(
        row["uncertain_worn"] for row in image_rows
    ),
    "suppressed_worn_predictions": sum(
        row["suppressed_worn"] for row in image_rows
    ),
    "limitations": [
        "This report evaluates every image in the locked test split at the confidence threshold selected on validation.",
        "A missed or borderless depiction region can leave an unverified false candidate; it must never be auto-counted by the application.",
        "Scene and depiction-surface gates pass only with complete reviewed metadata and positive image coverage in all six required strata.",
        "In locked-test accounting, an unverified model class-0 hard_hat_worn prediction with any positive overlap against a ground-truth model class-1 depiction_region box cannot match a real-wearer box and remains false.",
        "The filter is a fail-closed application policy and is not a detector guarantee.",
    ],
    "hardhat_class_metrics_after_poster_filter": class_metrics,
    "scene_metrics_by_class": scene_metrics,
}
summary_bytes = (
    json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
).encode("utf-8")
(REPORT_DIR / "summary.json").write_bytes(summary_bytes)

print("Full locked-test report:", REPORT_DIR)
print("Locked-test hardhat ground-truth boxes:", locked_test_hardhat_ground_truth_count, "(minimum:", MIN_LOCKED_TEST_HARDHAT_GROUND_TRUTH, ")")
print("Deployment gate:", deployment_gate_status)
print("Deployment gate reasons:", deployment_gate_reasons)
print("Per-surface/per-scene deployment gates:")
print(json.dumps(summary["depiction_surface_test_gates"], ensure_ascii=False, indent=2))
print("Uncertain hardhat candidates:", summary["uncertain_worn_predictions"])
print("Suppressed hardhat candidates:", summary["suppressed_worn_predictions"])

if deployment_gate_status != "approved":
    RELEASE_MANIFEST_PATH.unlink(missing_ok=True)
    RELEASE_MANIFEST_TEMP.unlink(missing_ok=True)
    DEPLOYMENT_TEST_SUMMARY_PATH.unlink(missing_ok=True)
    DEPLOYMENT_TEST_SUMMARY_TEMP.unlink(missing_ok=True)
    raise RuntimeError(
        "Deployment release gate is blocked; evaluation report was saved at "
        f"{REPORT_DIR}. Reasons: {deployment_gate_reasons}"
    )

summary_sha256 = hashlib.sha256(summary_bytes).hexdigest()
release_manifest = {
    "format": "helmet-vision-model-release-v9",
    "checkpoint_sha256": checkpoint_sha256,
    "class_names": CLASS_NAMES,
    "depiction_surface_gate_status": "pass_reviewed_metadata",
    "required_depiction_surface_types": list(CORE_DEPICTION_SURFACE_TYPES),
    "scene_metadata_source": "reviewed_metadata_csv",
    "dataset_archive_sha256": archive_sha256,
    "test_summary_sha256": summary_sha256,
    "hard_negative_source_groups_sha256": current_source_groups_sha256,
    "data_sufficiency_gate_status": data_sufficiency_gate_status,
    "validation_threshold_evidence_sha256": validation_threshold_evidence_sha256,
    "locked_test_start_marker_sha256": LOCKED_TEST_START_MARKER_SHA256,
    "required_hat_type_negative_sources": list(HAT_TYPE_NEGATIVE_REJECTION_SOURCES),
    "hat_type_negative_rejection_gate_status": hat_type_negative_rejection_gate_status,
}
manifest_bytes = (
    json.dumps(release_manifest, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
).encode("utf-8")
try:
    DEPLOYMENT_TEST_SUMMARY_TEMP.write_bytes(summary_bytes)
    RELEASE_MANIFEST_TEMP.write_bytes(manifest_bytes)
    DEPLOYMENT_TEST_SUMMARY_TEMP.replace(DEPLOYMENT_TEST_SUMMARY_PATH)
    RELEASE_MANIFEST_TEMP.replace(RELEASE_MANIFEST_PATH)
except Exception:
    RELEASE_MANIFEST_PATH.unlink(missing_ok=True)
    RELEASE_MANIFEST_TEMP.unlink(missing_ok=True)
    DEPLOYMENT_TEST_SUMMARY_PATH.unlink(missing_ok=True)
    DEPLOYMENT_TEST_SUMMARY_TEMP.unlink(missing_ok=True)
    raise
print("Deployment test summary:", DEPLOYMENT_TEST_SUMMARY_PATH)
print("Deployment release manifest:", RELEASE_MANIFEST_PATH)


## Checklist rà soát trước khi phát hành

- Xem ảnh dự đoán trên test với poster, ảnh in và màn hình, trong cả cảnh chỉ có hình minh họa và cảnh có người thật đứng cạnh.
- Kiểm tra người trong poster không được tự động tính. Tâm trong bất kỳ vùng poster hoặc coverage trên hợp các `depiction_region` từ 50% phải bị suppression; overlap dương dưới 50% khi tâm ở ngoài phải vào `uncertain` và không tính.
- Đo riêng false unverified hardhat candidates cho từng tổ hợp poster/ảnh in/màn hình × `depiction_only`/`mixed_scene`; cả sáu strata phải có ảnh và số false phải bằng 0.
- Chọn ngưỡng trên validation. Không điều chỉnh ngưỡng sau khi nhìn test.
- Tập test phải có metadata đã rà soát bao phủ chính xác mọi ảnh và đủ sáu strata surface/scene bắt buộc.
- Chỉ dùng checkpoint khi cell test pass và có `deployment_release.json` cùng `deployment_test_summary.json` cạnh thư mục run. Nếu gate fail, báo cáo chẩn đoán vẫn được lưu nhưng hai artifact triển khai bị xóa.
- Với từng subtype `traffic_helmet_worn`, `cap_worn`, `beanie_worn`, locked test cần ≥100 group subtype-only sạch ở mọi ảnh; tại threshold khóa trên validation yêu cầu 0 hộp và 0 ảnh có dự đoán `hard_hat_worn`. Gate phải pass trước khi cấp receipt v9.
- Receipt v9 ràng buộc SHA-256 của `best.pt`, thứ tự lớp, metadata test đã rà soát, mapping ID/type/box của từng surface vật lý, các surface bắt buộc, SHA-256 archive dataset và SHA-256 chính xác của summary test.
- Đưa ca thiếu ngữ cảnh vào hàng đợi `uncertain` của ứng dụng. Không tạo lớp detector uncertain hoặc suy ra rằng người trong poster là người thật.

Notebook có thể huấn luyện và đánh giá trên Colab, nhưng chưa có bộ dữ liệu hardhat-only đi kèm. Chất lượng của kết quả phụ thuộc vào việc gán nhãn poster và vùng hiển thị đầy đủ theo quy tắc trên.
